# ARGO VPP 2-DOF

Notebook reconstruit pour un Velocity Prediction Program stationnaire en eau calme a deux degres de liberte:

\[
x = \begin{bmatrix} V_B \\ \phi \end{bmatrix}
\]

Les deux equations resolues sont:

\[
F_\mathrm{drive} - R_\mathrm{tot} = 0
\]

\[
M_\mathrm{heel} - M_\mathrm{right} = 0
\]

**Modele 2-DOF : l'angle et la vitesse de derive ne sont pas calcules.  
La force hydrodynamique laterale est imposee egale a la force aerodynamique.**

Conventions internes:

- \(x\): axe longitudinal, positif de la poupe vers la proue.
- \(y\): axe transversal, positif vers babord.
- \(z\): axe vertical, positif vers le haut.
- Toutes les grandeurs internes sont en SI.
- Le nombre de Froude du nom STL n'est jamais utilise comme constante; \(Fn = V_B / \sqrt{g L_{WL}}\) est recalcule a chaque evaluation.

Ressources documentaires attendues:

- `Thesis_Pedro_Teixeira_104081_V3.pdf`
- `ORC-VPP-Documentation-2026.pdf`

Si ces PDF ne sont pas dans le dossier de travail, le notebook l'indique explicitement et applique les formulations codifiees dans les cellules ci-dessous.

> **Organisation du code :** les calculs réutilisables du notebook sont répartis dans les modules `fonctions/argo_vpp_nb_*.py`. Les cellules de saisie, widgets et graphiques restent dans le notebook et conservent leur ordre d’exécution.


## 1. Presentation du modele et hypotheses

Le modele est stationnaire, sans vagues, sans planage et sans accelerations. L'equilibre lateral n'est pas une equation du solveur: la force laterale hydrodynamique est imposee egale a la force laterale aerodynamique et sert uniquement aux termes de trainee induite et de moment hydrodynamique inclinant.

Le moment redressant est pre-calcule sous forme de courbe \(GZ(\phi)\), puis interpole pendant le solveur. Une correction dynamique OceanVPP `RMV` peut etre ajoutee.

In [2]:
from pathlib import Path

EXPECTED_DOCUMENTS = [
    Path("Thesis_Pedro_Teixeira_104081_V3.pdf"),
    Path("ORC-VPP-Documentation-2026.pdf"),
]

for document in EXPECTED_DOCUMENTS:
    if document.exists():
        print(f"Document disponible: {document}")
    else:
        print(f"AVERTISSEMENT: document introuvable dans le dossier courant: {document}")

Document disponible: Thesis_Pedro_Teixeira_104081_V3.pdf
Document disponible: ORC-VPP-Documentation-2026.pdf


## 2. Installation conditionnelle et imports

In [3]:
import importlib.util
import subprocess
import sys
import json
import warnings
from dataclasses import asdict, dataclass, field, fields
from math import pi
from typing import Any, Callable

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.optimize as opt
import trimesh
from IPython.display import clear_output, display
from matplotlib.collections import PolyCollection
from matplotlib.ticker import MaxNLocator
from scipy.interpolate import interp1d
from shapely.geometry import MultiPoint, MultiPolygon, Point, Polygon
from shapely.ops import triangulate, unary_union

# Global sweep configuration. *_STEPS counts intervals; endpoints are included.
TWA_RANGE_MIN_DEG = 30.0
TWA_RANGE_MAX_DEG = 150.0
TWA_RANGE_STEPS = 5
HEEL_RANGE_MIN_DEG = 0.0
HEEL_RANGE_MAX_DEG = 60.0
HEEL_RANGE_STEPS = 10


def analysis_angle_values(min_deg: float, max_deg: float, steps: int) -> np.ndarray:
    """Return an inclusive angle sweep defined by min/max and interval count."""

    if not np.isfinite(min_deg) or not np.isfinite(max_deg) or max_deg < min_deg:
        raise ValueError(f"Invalid angle range: min={min_deg}, max={max_deg}.")
    if int(steps) != steps or steps < 1:
        raise ValueError(f"Angle sweep steps must be a positive integer; got {steps!r}.")
    return np.linspace(float(min_deg), float(max_deg), int(steps) + 1)


plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

## 3. Constantes, dataclasses et conventions

Constantes physiques utilisees:

| Grandeur | Valeur |
|---|---:|
| `RHO_AIR` | 1.225 kg/m3 |
| `MU_AIR` | 1.81e-5 Pa.s |
| `RHO_WATER` | 1025.0 kg/m3 |
| `NU_WATER` | 1.19e-6 m2/s |
| `G` | 9.80665 m/s2 |
| `KNOT_TO_MPS` | 0.514444 |
| `RHO_CARBON` | 1500.0 kg/m3 |

In [4]:
from fonctions.argo_vpp_nb_models import *


## 4. Lecture des tables Delft et des fichiers polaires NACA

In [5]:
from fonctions.argo_vpp_nb_tables import *


## 5. Lecture, controle et orientation du STL

In [6]:
from fonctions.argo_vpp_nb_mesh import *

def set_3d_axes_equal(ax: Any, points: np.ndarray) -> None:
    """Set equal scale on 3D axes."""

    pts = np.asarray(points, dtype=float)
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float(np.max(maxs - mins))
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)

def plot_3d_control(state: BoatGeometryState | None = None, hull_mesh: trimesh.Trimesh | None = None, waterline_z: float | None = None) -> None:
    """Plot a 3D control view with hull, axes, water plane, CG, CB, wing, centerboard, rudder and ballast."""

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    meshes: list[tuple[str, trimesh.Trimesh, str, float]] = []
    if state is not None:
        meshes = [
            ("coque", hull_mesh if hull_mesh is not None else state.hull_mesh, "lightsteelblue", 0.35),
            ("aile", state.wing.mesh, "tab:orange", 0.55),
            ("derive", state.centerboard.mesh, "tab:green", 0.55),
            ("safran", state.rudder.mesh, "tab:red", 0.55),
        ]
        waterline_z = state.hydro.waterline_z
    elif hull_mesh is not None:
        meshes = [("coque", hull_mesh, "lightsteelblue", 0.35)]

    all_points = []
    for label, mesh, color, alpha in meshes:
        triangles = mesh.triangles
        collection = PolyCollection(triangles[:, :, [0, 1]], alpha=alpha, facecolor=color, edgecolor="k", linewidth=0.1)
        ax.add_collection3d(collection, zs=triangles[:, :, 2].mean(axis=1), zdir="z")
        all_points.append(mesh.vertices)

    if state is not None:
        G = state.masses.center_of_gravity
        B = safe_mesh_center(state.immersed_hull_mesh)
        ballast = state.masses.ballast_position
        ax.scatter([G[0]], [G[1]], [G[2]], c="black", s=60, label="G")
        ax.scatter([B[0]], [B[1]], [B[2]], c="blue", s=60, label="B")
        ax.scatter([ballast[0]], [ballast[1]], [ballast[2]], c="purple", s=60, label="ballast ponctuel")
        all_points += [G.reshape(1, 3), B.reshape(1, 3), ballast.reshape(1, 3)]

    if all_points:
        pts = np.vstack(all_points)
        x0, x1 = float(pts[:, 0].min()), float(pts[:, 0].max())
        y0, y1 = float(pts[:, 1].min()), float(pts[:, 1].max())
        if waterline_z is not None:
            xx, yy = np.meshgrid(np.linspace(x0, x1, 2), np.linspace(y0, y1, 2))
            zz = np.full_like(xx, waterline_z)
            ax.plot_surface(xx, yy, zz, color="cyan", alpha=0.18)
        ax.quiver(x0, y0, pts[:, 2].min(), 0.3 * (x1 - x0), 0, 0, color="r", label="x proue")
        ax.quiver(x0, y0, pts[:, 2].min(), 0, 0.3 * max(y1 - y0, 1e-6), 0, color="g", label="y babord")
        ax.quiver(x0, y0, pts[:, 2].min(), 0, 0, 0.3 * max(np.ptp(pts[:, 2]), 1e-6), color="b", label="z haut")
        ax.text(x0, 0, pts[:, 2].min(), "poupe")
        ax.text(x1, 0, pts[:, 2].min(), "proue")
        set_3d_axes_equal(ax, pts)

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.legend(loc="best")
    plt.show()


In [7]:
def plot_3d_control(state: BoatGeometryState | None = None, hull_mesh: trimesh.Trimesh | None = None, waterline_z: float | None = None) -> None:
    """Display an interactive 3D control view of the boat geometry."""

    fig = go.Figure()
    meshes: list[tuple[str, trimesh.Trimesh, str, float]] = []
    if state is not None:
        meshes = [
            ("coque", hull_mesh if hull_mesh is not None else state.hull_mesh, "lightsteelblue", 0.35),
            ("aile", state.wing.mesh, "darkorange", 0.55),
            ("derive", state.centerboard.mesh, "seagreen", 0.55),
            ("safran", state.rudder.mesh, "firebrick", 0.55),
        ]
        waterline_z = state.hydro.waterline_z
    elif hull_mesh is not None:
        meshes = [("coque", hull_mesh, "lightsteelblue", 0.35)]

    all_points: list[np.ndarray] = []
    for label, mesh, color, opacity in meshes:
        vertices = np.asarray(mesh.vertices, dtype=float)
        faces = np.asarray(mesh.faces, dtype=int)
        fig.add_trace(go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=color,
            opacity=opacity,
            name=label,
            flatshading=True,
            lighting={"ambient": 0.55, "diffuse": 0.8, "specular": 0.25, "roughness": 0.75},
            hovertemplate=f"{label}<extra></extra>",
        ))
        all_points.append(vertices)

    if state is not None:
        G = np.asarray(state.masses.center_of_gravity, dtype=float)
        B = np.asarray(safe_mesh_center(state.immersed_hull_mesh), dtype=float)
        ballast = np.asarray(state.masses.ballast_position, dtype=float)
        fig.add_trace(go.Scatter3d(
            x=[G[0], B[0], ballast[0]],
            y=[G[1], B[1], ballast[1]],
            z=[G[2], B[2], ballast[2]],
            mode="markers+text",
            text=["G", "B", "ballast ponctuel"],
            textposition="top center",
            marker={"size": 5, "color": ["black", "blue", "purple"]},
            name="Centres et ballast",
        ))
        all_points.extend([G.reshape(1, 3), B.reshape(1, 3), ballast.reshape(1, 3)])

    if all_points:
        points = np.vstack(all_points)
        x0, x1 = float(points[:, 0].min()), float(points[:, 0].max())
        y0, y1 = float(points[:, 1].min()), float(points[:, 1].max())
        z0, z1 = float(points[:, 2].min()), float(points[:, 2].max())
        dx = max(x1 - x0, 1e-6)
        dy = max(y1 - y0, 1e-6)
        dz = max(z1 - z0, 1e-6)
        if waterline_z is not None:
            xx, yy = np.meshgrid([x0, x1], [y0, y1])
            zz = np.full_like(xx, waterline_z)
            fig.add_trace(go.Surface(
                x=xx,
                y=yy,
                z=zz,
                colorscale=[[0, "cyan"], [1, "cyan"]],
                opacity=0.18,
                showscale=False,
                hoverinfo="skip",
                name="Plan d'eau",
            ))
        axis_start = [x0, y0, z0]
        axes = [
            ([x0 + 0.3 * dx, y0, z0], "red", "x proue"),
            ([x0, y0 + 0.3 * dy, z0], "green", "y babord"),
            ([x0, y0, z0 + 0.3 * dz], "blue", "z haut"),
        ]
        for endpoint, color, label in axes:
            fig.add_trace(go.Scatter3d(
                x=[axis_start[0], endpoint[0]],
                y=[axis_start[1], endpoint[1]],
                z=[axis_start[2], endpoint[2]],
                mode="lines+text",
                text=["", label],
                line={"color": color, "width": 5},
                name=label,
            ))
        fig.add_trace(go.Scatter3d(
            x=[x0, x1], y=[0, 0], z=[z0, z0],
            mode="text", text=["poupe", "proue"],
            showlegend=False, hoverinfo="skip",
        ))

    fig.update_layout(
        title="Controle 3D de la geometrie",
        height=700,
        margin={"l": 0, "r": 0, "b": 0, "t": 40},
        scene={
            "xaxis_title": "x [m]",
            "yaxis_title": "y [m]",
            "zaxis_title": "z [m]",
            "aspectmode": "data",
        },
        legend={"orientation": "h", "y": 1.02, "x": 0},
    )
    display(fig)

## 6. Decoupage de la coque par le plan d'eau

Le tirant d'eau de coque nue \(T_C\) est mesure depuis le point le plus bas du STL:

\[
z_{WL} = z_{\min,\mathrm{coque}} + T_C
\]

In [8]:
from fonctions.argo_vpp_nb_mesh import *


## 7. Calcul des parametres hydrostatiques de la coque

In [9]:
from fonctions.argo_vpp_nb_hydrostatics import *


## 8. Generation 3D de l'aile, de la derive et du safran

In [10]:
from fonctions.argo_vpp_nb_appendages import *


## 9. Calcul des masses et du centre de gravite

In [11]:
from fonctions.argo_vpp_nb_mass import *


## 10. Calcul de la courbe GZ

In [12]:
from fonctions.argo_vpp_nb_stability import *


## 11. Modele aerodynamique de l'aile rigide

Le centre d'effort est calcule a partir de la geometrie de l'aile. L'angle d'attaque est impose par l'interface, et l'orientation mecanique affichee vaut `AWA - alpha`.

In [13]:
from fonctions.argo_vpp_nb_aerodynamics import *


## 12. Modele hydrodynamique Delft

In [14]:
from fonctions.argo_vpp_nb_hydrodynamics import *


## 13. Equations d'equilibre 2-DOF

In [15]:
from fonctions.argo_vpp_nb_solver import *


## 14. Solveur numerique

In [16]:
# Les fonctions de calcul de cette section sont dans argo_vpp_nb_solver.py.


In [17]:
# Les fonctions de calcul de cette section sont dans argo_vpp_nb_solver.py.


In [18]:
from fonctions.argo_vpp_nb_solver import *

def build_manual_input_widgets() -> widgets.Tab:
    """Small ipywidgets interface for the manual Python-VPP style input."""

    current = {"manual_input": clone_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)}
    out = widgets.Output()
    def with_dimension_hint(control, explanation: str) -> widgets.HBox:
        hint = widgets.HTML(value=f"<small style='color:#666'>{explanation}</small>", layout=widgets.Layout(width="420px"))
        return widgets.HBox([control, hint], layout=widgets.Layout(align_items="center", width="100%"))
    json_path = widgets.Text(value="manual_vpp_input.json", description="JSON", layout=widgets.Layout(width="70%"))
    csv_path = widgets.Text(value=str(Path("Résultats VPP") / "ARGO_VPP_manual_results.csv"), description="CSV", layout=widgets.Layout(width="70%"))
    mode = widgets.Dropdown(options=["manual", "stl"], value="manual", description="Geometry mode")
    solver_method = widgets.Dropdown(options=["least_squares_multistart", "least_squares_single"], value="least_squares_multistart", description="Solver")
    sail_source = widgets.Dropdown(options=["orc", "hazen", "naca"], value="orc", description="Sail data")
    aero_model = widgets.Dropdown(options=["naca_wing", "pedro_hazen", "pedro_orc_low", "pedro_orc_high", "pedro_orc_medium"], value="pedro_orc_low", description="Aero model")
    hydro_model = widgets.Dropdown(options=[("oceanvpp_delft", "oceanvpp_delft"), ("orc_simplified_not_full_orc", "orc_simplified")], value="oceanvpp_delft", description="Hydro")
    model_note = widgets.HTML(value="<small>Aero model drives Sail data. ORC simplified is not the full ORC 2026 VPP.</small>")
    mass_mode = widgets.Dropdown(options=["fixed_loading", "design_ballast"], value="fixed_loading", description="Mass mode")
    wind_gradient = widgets.Checkbox(value=False, description="Use wind gradient")
    angle_ranges = widgets.HTML(value=(f"<b>TWA:</b> {TWA_RANGE_MIN_DEG:g} to {TWA_RANGE_MAX_DEG:g} deg ({TWA_RANGE_STEPS} steps)<br>"
                                      f"<b>Heel:</b> {HEEL_RANGE_MIN_DEG:g} to {HEEL_RANGE_MAX_DEG:g} deg ({HEEL_RANGE_STEPS} steps)"))
    tws = widgets.FloatRangeSlider(value=[8.0, 12.0], min=0.0, max=40.0, step=0.5, description="TWS", layout=widgets.Layout(width="90%"))
    tws_unit = widgets.Dropdown(options=["knots", "m/s"], value="knots", description="TWS unit")
    Hs = widgets.FloatText(value=0.0, description="Hs [m]")
    Ts = widgets.FloatText(value=0.0, description="Ts [s]")
    yacht_name = widgets.Text(value="Daring", description="Name")
    Lwl = widgets.FloatText(value=7.01, description="Lwl [m]")
    volume = widgets.FloatText(value=1.95, description="Volume [m3]")
    Bwl = widgets.FloatText(value=1.70, description="Bwl [m]")
    Tc = widgets.FloatText(value=0.45, description="Tc [m]")
    Swet = widgets.FloatText(value=11.5, description="Swet [m2]")
    Tmax = widgets.FloatText(value=1.35, description="Tmax [m]")
    Amax = widgets.FloatText(value=0.38, description="Amax [m2]")
    dmass = widgets.FloatText(value=2000.0, description="Delta [kg]")
    Ff = widgets.FloatText(value=0.75, description="Ff [m]")
    Fa = widgets.FloatText(value=0.55, description="Fa [m]")
    Boa = widgets.FloatText(value=1.98, description="Boa [m]")
    Loa = widgets.FloatText(value=9.9, description="Loa [m]")
    ks = widgets.FloatText(value=150.0, description="ks [um]")
    LCB = widgets.FloatText(value=-4.0, description="LCB %")
    sail_type = widgets.Dropdown(options=["main", "main+jib", "main+spinnaker", "rigid_wing"], value="main", description="Sail type")
    sail_name = widgets.Text(value="MN1", description="Name")
    P = widgets.FloatText(value=10.8, description="P [m]")
    E = widgets.FloatText(value=3.3, description="E [m]")
    roach = widgets.FloatText(value=0.1, description="Roach")
    BAD = widgets.FloatText(value=0.8, description="BAD [m]")
    naca = widgets.Text(value="NACA0012", description="NACA")
    alpha_min = widgets.FloatText(value=-5.0, description="alpha min")
    alpha_max = widgets.FloatText(value=20.0, description="alpha max")
    alpha_step = widgets.FloatText(value=1.0, description="alpha step")
    rig_I = widgets.FloatText(value=10.2, description="I [m]")
    rig_J = widgets.FloatText(value=3.0, description="J [m]")
    rig_LPG = widgets.FloatText(value=3.3, description="LPG [m]")
    rig_SL = widgets.FloatText(value=0.0, description="SL [m]")
    rig_EHM = widgets.FloatText(value=11.6, description="EHM [m]")
    rig_EMDC = widgets.FloatText(value=0.0, description="EMDC [m]")
    rig_HBI = widgets.FloatText(value=0.0, description="HBI [m]")
    rig_BAS = widgets.FloatText(value=0.0, description="BAS [m]")
    rig_fractionality = widgets.FloatText(value=0.0, description="fractionality")
    sailset_widget = widgets.Dropdown(options=["main_only", "main_jib", "main_spinnaker"], value="main_only", description="Sail set")
    spinnaker_type = widgets.Dropdown(options=["symmetric", "asymmetric"], value="symmetric", description="Spin type")
    reference_area_mode = widgets.Dropdown(options=["active_area", "pedro_nominal"], value="active_area", description="A ref")
    flat = widgets.FloatText(value=1.0, description="flat")
    reef = widgets.FloatText(value=1.0, description="reef")
    optimize_flat = widgets.Checkbox(value=False, description="Auto-optimize flat")
    optimize_reef = widgets.Checkbox(value=False, description="Auto-optimize reef")
    flat_min = widgets.FloatText(value=0.42, description="flat min")
    reef_min = widgets.FloatText(value=0.70, description="reef min")
    pedro_hardcoded_hydroheel = widgets.Checkbox(value=True, description="Pedro hydroheel")
    keel_type = widgets.Dropdown(options=["short", "long", "fin", "centerboard", "daggerboard"], value="short", description="Keel")
    Lkeel = widgets.FloatText(value=1.2, description="Lkeel [m]")
    Dkeel = widgets.FloatText(value=0.9, description="Dkeel [m]")
    keel_tc = widgets.FloatText(value=0.15, description="t/c")
    keel_mass = widgets.FloatText(value=0.0, description="Mass [kg]")
    rud_Cu = widgets.FloatText(value=0.32, description="Cu [m]")
    rud_Cl = widgets.FloatText(value=0.18, description="Cl [m]")
    rud_b = widgets.FloatText(value=0.75, description="b [m]")
    gm = widgets.FloatText(value=0.55, description="GM [m]")
    delft_folder = widgets.Text(value=str(default_delft_folder()), description="Tables", layout=widgets.Layout(width="90%"))
    polar_folder = widgets.Text(value="Profil NACA", description="Polars", layout=widgets.Layout(width="90%"))

    selection_sync = {"active": False}
    sail_configuration_sync = {"active": False}
    sail_type_to_set = {"main": "main_only", "main+jib": "main_jib", "main+spinnaker": "main_spinnaker", "rigid_wing": "main_only"}
    sail_set_to_type = {"main_only": "main", "main_jib": "main+jib", "main_spinnaker": "main+spinnaker"}

    def sailset_requires_depowering(value: str) -> bool:
        return str(value) in {"main_jib", "main_spinnaker"}

    def sync_sailset_from_sail_type(change: dict[str, Any]) -> None:
        if sail_configuration_sync["active"]:
            return
        new_set = sail_type_to_set.get(change["new"])
        if new_set is None:
            return
        sail_configuration_sync["active"] = True
        try:
            sailset_widget.value = new_set
            if sailset_requires_depowering(new_set):
                optimize_flat.value = True
                optimize_reef.value = True
        finally:
            sail_configuration_sync["active"] = False

    def sync_sail_type_from_sailset(change: dict[str, Any]) -> None:
        if sail_configuration_sync["active"]:
            return
        new_type = sail_set_to_type.get(change["new"])
        if new_type is None:
            return
        sail_configuration_sync["active"] = True
        try:
            sail_type.value = new_type
            if sailset_requires_depowering(change["new"]):
                optimize_flat.value = True
                optimize_reef.value = True
        finally:
            sail_configuration_sync["active"] = False

    def sync_from_aero(change: dict[str, Any]) -> None:
        if selection_sync["active"]:
            return
        selection_sync["active"] = True
        try:
            sail_source.value = AERO_MODEL_TO_SAIL_DATA[change["new"]]
            pedro_hardcoded_hydroheel.value = is_pedro_aero_model(change["new"])
        finally:
            selection_sync["active"] = False

    def sync_from_sail_data(change: dict[str, Any]) -> None:
        if selection_sync["active"]:
            return
        selection_sync["active"] = True
        try:
            aero_model.value = SAIL_DATA_TO_DEFAULT_AERO_MODEL[change["new"]]
            pedro_hardcoded_hydroheel.value = is_pedro_aero_model(aero_model.value)
        finally:
            selection_sync["active"] = False

    aero_model.observe(sync_from_aero, names="value")
    sail_source.observe(sync_from_sail_data, names="value")
    sail_type.observe(sync_sailset_from_sail_type, names="value")
    sailset_widget.observe(sync_sail_type_from_sailset, names="value")

    def apply_manual_input(mi: ManualVPPInput) -> None:
        mi = normalize_aero_selection(mi)
        mi.environment.twa_min_deg = TWA_RANGE_MIN_DEG
        mi.environment.twa_max_deg = TWA_RANGE_MAX_DEG
        mi.environment.twa_steps = TWA_RANGE_STEPS
        current["manual_input"] = clone_manual_input(mi)
        mode.value = mi.advanced.geometry_input_mode
        solver_method.value = mi.solver.solver_method
        sail_source.value = mi.solver.sail_coefficient_data_source
        aero_model.value = mi.advanced.aero_model
        hydro_model.value = mi.advanced.hydro_model
        mass_mode.value = mi.advanced.mass_mode
        wind_gradient.value = mi.advanced.use_wind_gradient
        tws.value = [mi.environment.tws_min, mi.environment.tws_max]
        tws_unit.value = mi.environment.tws_unit
        Hs.value = mi.environment.Hs_m
        Ts.value = mi.environment.Ts_s
        yacht_name.value = mi.yacht.name
        Lwl.value = mi.yacht.Lwl_m
        volume.value = mi.yacht.displaced_volume_m3
        Bwl.value = mi.yacht.Bwl_m
        Tc.value = mi.yacht.Tc_m
        Swet.value = mi.yacht.Swet_m2
        Tmax.value = mi.yacht.Tmax_m
        Amax.value = mi.yacht.Amax_m2
        dmass.value = mi.yacht.displacement_mass_kg
        Ff.value = mi.yacht.Ff_m
        Fa.value = mi.yacht.Fa_m
        Boa.value = mi.yacht.Boa_m
        Loa.value = mi.yacht.Loa_m
        ks.value = mi.yacht.hull_roughness_ks_um
        LCB.value = mi.yacht.LCB_pct_midship
        sail_type.value = mi.main_sail.main_sail_type
        sail_name.value = mi.main_sail.name
        P.value = mi.main_sail.P_m
        E.value = mi.main_sail.E_m
        roach.value = mi.main_sail.roach
        BAD.value = mi.main_sail.BAD_m
        naca.value = mi.main_sail.naca_profile
        alpha_min.value = mi.main_sail.alpha_min_deg
        alpha_max.value = mi.main_sail.alpha_max_deg
        alpha_step.value = mi.main_sail.alpha_step_deg
        rig_I.value = mi.rig.I_m
        rig_J.value = mi.rig.J_m
        rig_LPG.value = mi.rig.LPG_m
        rig_SL.value = mi.rig.SL_m
        rig_EHM.value = mi.rig.EHM_m if mi.rig.EHM_m is not None else 0.0
        rig_EMDC.value = mi.rig.EMDC_m if mi.rig.EMDC_m is not None else 0.0
        rig_HBI.value = mi.rig.HBI_m
        rig_BAS.value = mi.rig.BAS_m if mi.rig.BAS_m is not None else 0.0
        rig_fractionality.value = mi.rig.fractionality if mi.rig.fractionality is not None else 0.0
        sailset_widget.value = mi.sailset.sailset
        spinnaker_type.value = mi.sailset.spinnaker_type
        reference_area_mode.value = mi.sailset.reference_area_mode
        flat.value = mi.sailset.flat
        reef.value = mi.sailset.reef
        optimize_flat.value = bool(mi.sailset.optimize_flat or mi.sailset.optimize_flat_reef or sailset_requires_depowering(mi.sailset.sailset))
        optimize_reef.value = bool(mi.sailset.optimize_reef or mi.sailset.optimize_flat_reef or sailset_requires_depowering(mi.sailset.sailset))
        flat_min.value = mi.sailset.flat_min
        reef_min.value = mi.sailset.reef_min
        pedro_hardcoded_hydroheel.value = mi.advanced.pedro_hardcoded_hydroheel
        keel_type.value = mi.keel.keel_type
        Lkeel.value = mi.keel.Lkeel_m
        Dkeel.value = mi.keel.Dkeel_m
        keel_tc.value = mi.keel.thickness_ratio
        keel_mass.value = mi.keel.mass_kg
        rud_Cu.value = mi.rudder.Cu_m
        rud_Cl.value = mi.rudder.Cl_m
        rud_b.value = mi.rudder.b_m
        gm.value = mi.advanced.manual_GM_m
        delft_folder.value = mi.advanced.delft_table_folder
        polar_folder.value = mi.advanced.polar_folder

    def collect_manual_input() -> ManualVPPInput:
        depowering_required = sailset_requires_depowering(sailset_widget.value)
        solver = SolverSettings(
            solver_method=solver_method.value,
            sail_coefficient_data_source=sail_source.value,
            geometry_input_mode=mode.value,
            hydro_model=hydro_model.value,
            mass_mode=mass_mode.value,
            use_wind_gradient=bool(wind_gradient.value),
        )
        return ManualVPPInput(
            solver=solver,
            environment=EnvironmentSettings(twa_min_deg=TWA_RANGE_MIN_DEG, twa_max_deg=TWA_RANGE_MAX_DEG, twa_steps=TWA_RANGE_STEPS, tws_min=float(tws.value[0]), tws_max=float(tws.value[1]), tws_unit=tws_unit.value, Hs_m=float(Hs.value), Ts_s=float(Ts.value)),
            yacht=YachtParticulars(name=yacht_name.value, Lwl_m=float(Lwl.value), displaced_volume_m3=float(volume.value), Bwl_m=float(Bwl.value), Tc_m=float(Tc.value), Swet_m2=float(Swet.value), Tmax_m=float(Tmax.value), Amax_m2=float(Amax.value), displacement_mass_kg=float(dmass.value), Ff_m=float(Ff.value), Fa_m=float(Fa.value), Boa_m=float(Boa.value), Loa_m=float(Loa.value), hull_roughness_ks_um=float(ks.value), LCB_pct_midship=float(LCB.value)),
            main_sail=MainSailInput(main_sail_type=sail_type.value, name=sail_name.value, P_m=float(P.value), E_m=float(E.value), roach=float(roach.value), BAD_m=float(BAD.value), naca_profile=naca.value, alpha_min_deg=float(alpha_min.value), alpha_max_deg=float(alpha_max.value), alpha_step_deg=float(alpha_step.value)),
            rig=RigInput(I_m=float(rig_I.value), J_m=float(rig_J.value), LPG_m=float(rig_LPG.value), SL_m=float(rig_SL.value), EHM_m=float(rig_EHM.value) if float(rig_EHM.value) > 0 else None, EMDC_m=float(rig_EMDC.value) if float(rig_EMDC.value) > 0 else None, HBI_m=float(rig_HBI.value), BAS_m=float(rig_BAS.value) if float(rig_BAS.value) > 0 else None, fractionality=float(rig_fractionality.value) if float(rig_fractionality.value) > 0 else None),
            sailset=SailSetInput(sailset=sailset_widget.value, spinnaker_type=spinnaker_type.value, reference_area_mode=reference_area_mode.value, flat=float(flat.value), reef=float(reef.value), optimize_flat=bool(optimize_flat.value or depowering_required), optimize_reef=bool(optimize_reef.value or depowering_required), flat_min=float(flat_min.value), reef_min=float(reef_min.value)),
            keel=KeelInput(keel_type=keel_type.value, Lkeel_m=float(Lkeel.value), Dkeel_m=float(Dkeel.value), thickness_ratio=float(keel_tc.value), mass_kg=float(keel_mass.value)),
            rudder=RudderInput(Cu_m=float(rud_Cu.value), Cl_m=float(rud_Cl.value), b_m=float(rud_b.value)),
            advanced=AdvancedARGOSettings(geometry_input_mode=mode.value, hydro_model=hydro_model.value, mass_mode=mass_mode.value, aero_model=aero_model.value, use_wind_gradient=bool(wind_gradient.value), delft_table_folder=delft_folder.value, polar_folder=polar_folder.value, manual_GM_m=float(gm.value), pedro_hardcoded_hydroheel=bool(pedro_hardcoded_hydroheel.value)),
        )

    load_daring = widgets.Button(description="Charger exemple Python-VPP Daring", button_style="")
    load_micro = widgets.Button(description="Charger exemple Microtransat", button_style="")
    preview_3d = widgets.Button(description="Afficher le modele 3D", icon="cube")
    preview_out = widgets.Output()
    calc = widgets.Button(description="Calculer", button_style="success")
    process = widgets.Button(description="Process Specifications", button_style="")
    import_json_button = widgets.Button(description="Importer JSON")
    export_json_button = widgets.Button(description="Exporter JSON")
    export_csv_button = widgets.Button(description="Exporter CSV")
    progress = widgets.IntProgress(value=0, min=0, max=1, description="0%", bar_style="", style={"description_width": "55px"}, layout=widgets.Layout(width="90%", display="none"))
    progress_status = widgets.HTML(value="", layout=widgets.Layout(width="90%", display="none"))

    def on_load_daring(_):
        apply_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)

    def on_load_micro(_):
        apply_manual_input(DEFAULT_MANUAL_MICROTRANSAT)

    def on_process(_):
        with out:
            clear_output()
            mi = collect_manual_input()
            display(pd.DataFrame([process_specifications(mi)]))

    def on_preview_3d(_):
        with preview_out:
            clear_output(wait=True)
            try:
                manual_input = collect_manual_input()
                state = build_model_from_manual_input(manual_input)
                preview_hull = build_manual_preview_hull(manual_input.yacht)
                display(widgets.HTML("<i>Aperçu indicatif d'une coque générée à partir des paramètres du formulaire ; la stabilité manuelle reste calculée avec GM.</i>"))
                plot_3d_control(
                    state,
                    hull_mesh=preview_hull,
                )
                plot_gz_curve(state)
            except Exception as exc:
                print(f"Erreur d'affichage du modele 3D: {type(exc).__name__}: {exc}")

    def on_calc(_):
        calc.disabled = True
        progress.layout.display = ""
        progress_status.layout.display = ""
        progress.value = 0
        progress.bar_style = ""
        progress.description = "0%"
        progress_status.value = "<b>Preparation du modele...</b>"

        def report_progress(completed: int, total: int, tws_value: float, twa_value: float, success: bool) -> None:
            progress.max = max(total, 1)
            progress.value = completed
            progress.description = f"{round(100 * completed / max(total, 1))}%"
            point_status = "OK" if success else "invalide"
            progress_status.value = f"Point {completed}/{total} - TWS {tws_value:g} {tws_unit.value}, TWA {twa_value:g} deg - {point_status}"

        try:
            with out:
                clear_output()
                mi = collect_manual_input()
                tws_values = [mi.environment.tws_min] if abs(mi.environment.tws_min - mi.environment.tws_max) < 1e-12 else [mi.environment.tws_min, mi.environment.tws_max]
                twa_values = analysis_angle_values(TWA_RANGE_MIN_DEG, TWA_RANGE_MAX_DEG, TWA_RANGE_STEPS)
                total_conditions = len(tws_values) * len(twa_values)
                progress.max = max(total_conditions, 1)
                state = build_model_from_manual_input(mi)
                APP_STATE["geometry"] = state
                APP_STATE["geometry_summary"] = geometry_summary_dataframe(state)
                progress_status.value = f"Calcul de {total_conditions} points..."
                df = solve_vpp_grid(tws_values, state, twa_values_deg=twa_values, tws_unit=mi.environment.tws_unit, progress_callback=report_progress)
                APP_STATE["results"] = df
                valid_count = int(df["solver_success"].fillna(False).sum())
                progress.value = progress.max
                progress.description = "100%"
                progress.bar_style = "success" if valid_count == total_conditions else "warning"
                progress_status.value = f"Termine - {valid_count}/{total_conditions} points valides. Generation des graphiques..."
                with pd.option_context("display.max_rows", None, "display.max_columns", None):
                    display(APP_STATE["geometry_summary"])
                display(df)
                plot_3d_control(state)
                plot_gz_curve(state)
                plot_all_results(df, state, include_gz=False)
                progress_status.value = f"Termine - {valid_count}/{total_conditions} points valides."
        except Exception as exc:
            progress.bar_style = "danger"
            progress.description = "Erreur"
            progress_status.value = f"<b>Calcul interrompu:</b> {type(exc).__name__}: {exc}"
            with out:
                print(f"Erreur de calcul: {type(exc).__name__}: {exc}")
        finally:
            calc.disabled = False

    def on_import_json(_):
        with out:
            clear_output()
            mi = manual_input_from_json(json_path.value)
            apply_manual_input(mi)
            print(f"Imported {json_path.value}")

    def on_export_json(_):
        with out:
            clear_output()
            path = manual_input_to_json(collect_manual_input(), json_path.value)
            print(f"Exported {path}")

    def on_export_csv(_):
        with out:
            clear_output()
            if APP_STATE.get("results") is None:
                print("No result to export.")
            else:
                path = export_vpp_results_csv(APP_STATE["results"], csv_path.value)
                print(f"Exported {path}")

    load_daring.on_click(on_load_daring)
    load_micro.on_click(on_load_micro)
    process.on_click(on_process)
    preview_3d.on_click(on_preview_3d)
    calc.on_click(on_calc)
    import_json_button.on_click(on_import_json)
    export_json_button.on_click(on_export_json)
    export_csv_button.on_click(on_export_csv)

    tabs = widgets.Tab(children=[
        widgets.VBox([mode, solver_method, sail_source, aero_model, hydro_model, model_note, mass_mode, wind_gradient, process]),
        widgets.VBox([angle_ranges, with_dimension_hint(tws, "Vitesse du vent vrai (TWS) ; plage de calcul, selon l'unite choisie."), tws_unit, with_dimension_hint(Hs, "Hauteur significative des vagues."), with_dimension_hint(Ts, "Periode des vagues." )]),
        widgets.VBox([yacht_name, with_dimension_hint(Lwl, "Longueur de coque a la flottaison."), with_dimension_hint(volume, "Volume de deplacement de la coque."), with_dimension_hint(Bwl, "Largeur de coque a la flottaison."), with_dimension_hint(Tc, "Tirant d'eau de la coque."), with_dimension_hint(Swet, "Surface mouillee de la coque."), with_dimension_hint(Tmax, "Tirant d'eau maximal (quille comprise)."), with_dimension_hint(Amax, "Surface maximale de la section immergee."), with_dimension_hint(dmass, "Masse de deplacement du bateau."), with_dimension_hint(Ff, "Franc-bord a l'avant."), with_dimension_hint(Fa, "Franc-bord a l'arriere."), with_dimension_hint(Boa, "Largeur maximale de la coque."), with_dimension_hint(Loa, "Longueur hors tout."), with_dimension_hint(ks, "Hauteur de rugosite de la coque."), with_dimension_hint(LCB, "Position longitudinale du centre de carene, en pourcentage par rapport au milieu du bateau.")]),
        widgets.VBox([sail_type, sail_name, with_dimension_hint(P, "Longueur de guindant de la grand-voile."), with_dimension_hint(E, "Longueur de bordure de la grand-voile."), with_dimension_hint(roach, "Surplus relatif de surface du au rond de chute, sans unite."), with_dimension_hint(BAD, "Decalage vertical du centre d'effort de la grand-voile."), naca, with_dimension_hint(alpha_min, "Angle d'attaque minimal explore par le solveur."), with_dimension_hint(alpha_max, "Angle d'attaque maximal explore par le solveur."), with_dimension_hint(alpha_step, "Pas angulaire du balayage de l'angle d'attaque." )]),
        widgets.VBox([sailset_widget, spinnaker_type, reference_area_mode, with_dimension_hint(rig_I, "Hauteur de reference du greement (I)."), with_dimension_hint(rig_J, "Base du triangle avant du greement (J)."), with_dimension_hint(rig_LPG, "Largeur de recouvrement du foc, mesuree perpendiculairement au mat."), with_dimension_hint(rig_SL, "Longueur de guindant du spi."), with_dimension_hint(rig_EHM, "Hauteur effective du mat utilisee par le modele."), with_dimension_hint(rig_EMDC, "Largeur effective du mat prise en compte pour sa trainee."), with_dimension_hint(rig_HBI, "Hauteur de reference ajoutee au calcul de hauteur du centre d'effort."), with_dimension_hint(rig_BAS, "Parametre BAS, utilise avec P pour calculer la fractionnalite du greement."), with_dimension_hint(rig_fractionality, "Fraction du greement, sans unite."), with_dimension_hint(flat, "Facteur de creux de la voile, sans unite (1 = valeur nominale)."), with_dimension_hint(reef, "Facteur de reduction de surface par ris, sans unite (1 = pleine surface)."), optimize_flat, with_dimension_hint(flat_min, "Borne inferieure autorisee pour le facteur de creux."), optimize_reef, with_dimension_hint(reef_min, "Borne inferieure autorisee pour le facteur de ris."), pedro_hardcoded_hydroheel]),
        widgets.VBox([keel_type, with_dimension_hint(Lkeel, "Longueur de la quille dans le sens longitudinal."), with_dimension_hint(Dkeel, "Profondeur (envergure verticale) de la quille."), with_dimension_hint(keel_tc, "Rapport epaisseur / corde de la quille, sans unite."), with_dimension_hint(keel_mass, "Masse de la quille." )]),
        widgets.VBox([with_dimension_hint(rud_Cu, "Corde du safran a sa partie superieure."), with_dimension_hint(rud_Cl, "Corde du safran a sa partie inferieure."), with_dimension_hint(rud_b, "Envergure verticale du safran.")]),
        widgets.VBox([with_dimension_hint(gm, "Hauteur metacentrique (GM), indicateur de stabilite initiale."), delft_folder, polar_folder, json_path, csv_path, widgets.HBox([load_daring, load_micro]), widgets.HBox([preview_3d, calc, import_json_button, export_json_button, export_csv_button]), preview_out, progress, progress_status, out]),
    ])
    for idx, title in enumerate(["Solver Settings", "Environment", "Yacht particulars", "Main Sail / Wing", "Pedro Aero", "Keel", "Rudder", "Advanced ARGO settings"]):
        tabs.set_title(idx, title)
    return tabs



## 15. Interface ipywidgets

In [19]:
APP_STATE: dict[str, Any] = {
    "geometry": None,
    "results": None,
    "geometry_summary": None,
}


def compute_true_T_max(meshes: list[trimesh.Trimesh], z_WL: float, eps: float = 1e-9) -> tuple[float, float]:
    """Compute true maximum draft from submerged hull, centerboard and rudder vertices."""

    submerged_z: list[np.ndarray] = []
    for mesh in meshes:
        vertices = np.asarray(mesh.vertices, dtype=float)
        if len(vertices) == 0:
            continue
        mask = vertices[:, 2] <= z_WL + eps
        if np.any(mask):
            submerged_z.append(vertices[mask, 2])
    if not submerged_z:
        raise ValueError("T_max impossible: aucun point immerge sous la flottaison.")
    z_min_submerged = float(np.min(np.concatenate(submerged_z)))
    T_max = float(z_WL - z_min_submerged)
    if not np.isfinite(T_max) or T_max <= 0:
        raise ValueError(f"T_max invalide: {T_max}.")
    return T_max, z_min_submerged


import base64
import pickle
from datetime import datetime, timezone

GEOMETRY_CACHE_MAGIC = "ARGO_VPP_GEOMETRY_CACHE_V1"


def save_geometry_cache(state: BoatGeometryState, path: str | Path) -> Path:
    """Save the complete analyzed geometry state in a portable text file."""

    if not isinstance(state, BoatGeometryState):
        raise TypeError("Only a BoatGeometryState can be saved.")
    path = Path(path)
    if path.suffix.lower() != ".txt":
        path = path.with_suffix(".txt")
    payload = {
        "format": GEOMETRY_CACHE_MAGIC,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "state": state,
    }
    encoded = base64.b64encode(pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL)).decode("ascii")
    header = json.dumps({"format": GEOMETRY_CACHE_MAGIC, "created_utc": payload["created_utc"]})
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(f"{GEOMETRY_CACHE_MAGIC}\n{header}\n{encoded}\n", encoding="ascii")
    return path


def load_geometry_cache(path: str | Path) -> BoatGeometryState:
    """Load a previously analyzed geometry state from a text file."""

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Fichier de geometrie introuvable: {path}")
    lines = path.read_text(encoding="ascii").splitlines()
    if len(lines) < 3 or lines[0].strip() != GEOMETRY_CACHE_MAGIC:
        raise ValueError("Format de cache geometrie invalide ou version non supportee.")
    try:
        metadata = json.loads(lines[1])
        payload = pickle.loads(base64.b64decode("".join(lines[2:])))
    except (ValueError, TypeError, json.JSONDecodeError, pickle.UnpicklingError) as exc:
        raise ValueError(f"Lecture du cache geometrie impossible: {exc}") from exc
    if metadata.get("format") != GEOMETRY_CACHE_MAGIC or payload.get("format") != GEOMETRY_CACHE_MAGIC:
        raise ValueError("Version du cache geometrie non supportee.")
    state = payload.get("state")
    if not isinstance(state, BoatGeometryState):
        raise ValueError("Le cache ne contient pas un BoatGeometryState valide.")
    return state


def build_widgets() -> widgets.Tab:
    """Create the requested ipywidgets tab interface."""

    def with_dimension_hint(control, explanation: str) -> widgets.HBox:
        hint = widgets.HTML(value=f"<small style='color:#666'>{explanation}</small>", layout=widgets.Layout(width="420px"))
        return widgets.HBox([control, hint], layout=widgets.Layout(align_items="center", width="100%"))

    stl_options = available_stl_files()
    default_stl_path = stl_options[0] if stl_options else GeometryInputs().stl_path
    stl_path = widgets.Dropdown(options=stl_options if stl_options else [default_stl_path], value=default_stl_path, description="STL", layout=widgets.Layout(width="80%"))
    refresh_stl = widgets.Button(description="Rafraichir STL", icon="refresh", layout=widgets.Layout(width="150px"))
    stl_unit = widgets.Dropdown(options=["auto", "m", "cm", "mm"], value=GeometryInputs().stl_unit, description="Unite STL")
    orientation = widgets.Dropdown(
        options=["x_min = poupe, x_max = proue", "x_min = proue, x_max = poupe"],
        value="x_min = poupe, x_max = proue",
        description="Orientation",
        layout=widgets.Layout(width="95%"),
    )
    draft = widgets.FloatText(value=GeometryInputs().hull_draft_m, description="T_C [m]")
    delft_folder = widgets.Text(value=str(default_delft_folder()), description="Tables", layout=widgets.Layout(width="95%"))
    polar_folder = widgets.Text(value="Profil NACA", description="Profils", layout=widgets.Layout(width="75%"))
    refresh_profiles = widgets.Button(description="Rafraichir NACA", icon="refresh", layout=widgets.Layout(width="165px"))

    wing_contour = widgets.Textarea(value=GeometryInputs().wing_contour_text, description="Contour aile", layout=widgets.Layout(width="95%", height="150px"))
    cb_contour = widgets.Textarea(value=GeometryInputs().centerboard_contour_text, description="Contour derive", layout=widgets.Layout(width="95%", height="130px"))
    rudder_contour = widgets.Textarea(value=GeometryInputs().rudder_contour_text, description="Contour safran", layout=widgets.Layout(width="95%", height="130px"))
    naca_options = available_naca_profiles(polar_folder.value)
    if not naca_options:
        naca_options = [("NACA0012 - profil manquant", "NACA0012")]
    naca_values = [value for _label, value in naca_options]
    def default_profile(*preferred: str) -> str:
        for code in preferred:
            if code in naca_values:
                return code
        return naca_values[0]
    wing_naca = widgets.Dropdown(options=naca_options, value=default_profile("NACA0012"), description="NACA aile")
    cb_naca = widgets.Dropdown(options=naca_options, value=default_profile("NACA0010", "NACA0012"), description="NACA derive")
    rudder_naca = widgets.Dropdown(options=naca_options, value=default_profile("NACA0009", "NACA0010", "NACA0008"), description="NACA safran")

    hull_t = widgets.FloatText(value=1.2, description="ep coque mm")
    cb_t = widgets.FloatText(value=1.0, description="ep derive mm")
    rudder_t = widgets.FloatText(value=0.9, description="ep safran mm")
    wing_t = widgets.FloatText(value=0.8, description="ep aile mm")
    ballast_mode = widgets.Dropdown(options=["automatique", "manuel"], value="automatique", description="Ballast")
    manual_ballast = widgets.FloatText(value=0.0, description="ballast kg")
    equip_mass = widgets.FloatText(value=0.0, description="equip kg")
    equip_x = widgets.FloatText(value=0.0, description="x equip")
    equip_y = widgets.FloatText(value=0.0, description="y equip")
    equip_z = widgets.FloatText(value=0.0, description="z equip")

    tws_range = widgets.IntRangeSlider(value=[1, 15], min=1, max=15, step=1, description="TWS kn")
    exclude_tws = widgets.SelectMultiple(options=list(range(1, 16)), value=(), description="Retirer")
    alpha_mode = widgets.Dropdown(options=["manuel", "automatique"], value=GeometryInputs().wing_alpha_mode, description="Mode alpha")
    alpha = widgets.FloatSlider(value=6.0, min=0.0, max=20.0, step=0.5, description="alpha deg")
    alpha_min = widgets.FloatSlider(value=0.0, min=0.0, max=20.0, step=0.5, description="alpha min")
    alpha_max = widgets.FloatSlider(value=15.0, min=0.0, max=20.0, step=0.5, description="alpha max")
    alpha_step = widgets.FloatSlider(value=1.0, min=0.25, max=5.0, step=0.25, description="pas alpha")
    oswald = widgets.FloatSlider(value=0.85, min=0.4, max=1.0, step=0.01, description="Oswald")
    auto_aero_model = widgets.Dropdown(options=[("NACA wing", "naca_wing"), ("Pedro Hazen", "pedro_hazen"), ("Pedro ORC low", "pedro_orc_low"), ("Pedro ORC high", "pedro_orc_high"), ("Pedro ORC medium", "pedro_orc_medium")], value="naca_wing", description="Aero model")
    auto_main_sail_P = widgets.FloatText(value=10.8, description="Main P [m]")
    auto_main_sail_E = widgets.FloatText(value=3.3, description="Main E [m]")
    auto_main_sail_BAD = widgets.FloatText(value=0.8, description="Main BAD [m]")
    auto_sailset = widgets.Dropdown(options=[("Main only", "main_only"), ("Main + jib", "main_jib"), ("Main + spinnaker", "main_spinnaker")], value="main_only", description="Sail set")
    auto_spinnaker_type = widgets.Dropdown(options=["symmetric", "asymmetric"], value="symmetric", description="Spin type")
    auto_reference_area_mode = widgets.Dropdown(options=["active_area", "pedro_nominal"], value="active_area", description="A ref")
    auto_rig_I = widgets.FloatText(value=10.2, description="Rig I [m]")
    auto_rig_J = widgets.FloatText(value=3.0, description="Rig J [m]")
    auto_rig_LPG = widgets.FloatText(value=3.3, description="Rig LPG [m]")
    auto_rig_SL = widgets.FloatText(value=0.0, description="Rig SL [m]")
    auto_rig_EHM = widgets.FloatText(value=11.6, description="Rig EHM [m]")
    auto_rig_EMDC = widgets.FloatText(value=0.0, description="Rig EMDC [m]")
    auto_rig_HBI = widgets.FloatText(value=0.0, description="Rig HBI [m]")
    auto_rig_BAS = widgets.FloatText(value=0.0, description="Rig BAS [m]")
    auto_rig_fractionality = widgets.FloatText(value=0.0, description="Fractionality")
    auto_hull_Ff = widgets.FloatText(value=0.75, description="Ff [m]")
    auto_hull_Fa = widgets.FloatText(value=0.55, description="Fa [m]")
    auto_hull_Boa = widgets.FloatText(value=1.98, description="Boa [m]")
    auto_flat = widgets.FloatText(value=1.0, description="flat")
    auto_reef = widgets.FloatText(value=1.0, description="reef")
    auto_optimize_flat = widgets.Checkbox(value=False, description="Auto-optimize flat")
    auto_flat_min = widgets.FloatText(value=0.42, description="flat min")
    auto_optimize_reef = widgets.Checkbox(value=False, description="Auto-optimize reef")
    auto_reef_min = widgets.FloatText(value=0.70, description="reef min")
    auto_pedro_hydroheel = widgets.Checkbox(value=True, description="Pedro hydroheel")
    auto_pedro_note = widgets.HTML(value="<small>Pedro uses apparent-wind coefficient tables; NACA alpha controls above are ignored in Pedro mode.</small>")
    auto_pedro_controls = widgets.VBox([
        auto_pedro_note,
        with_dimension_hint(auto_main_sail_P, "Longueur de guindant de la grand-voile."),
        with_dimension_hint(auto_main_sail_E, "Longueur de bordure de la grand-voile."),
        with_dimension_hint(auto_main_sail_BAD, "Decalage vertical du centre d'effort de la grand-voile."),
        auto_sailset, auto_spinnaker_type, auto_reference_area_mode,
        with_dimension_hint(auto_rig_I, "Hauteur de reference du greement (I)."),
        with_dimension_hint(auto_rig_J, "Base du triangle avant du greement (J)."),
        with_dimension_hint(auto_rig_LPG, "Largeur de recouvrement du foc, perpendiculaire au mat."),
        with_dimension_hint(auto_rig_SL, "Longueur de guindant du spi."),
        with_dimension_hint(auto_rig_EHM, "Hauteur effective du mat utilisee par le modele."),
        with_dimension_hint(auto_rig_EMDC, "Largeur effective du mat prise en compte pour sa trainee."),
        with_dimension_hint(auto_rig_HBI, "Hauteur de reference ajoutee au calcul de hauteur du centre d'effort."),
        with_dimension_hint(auto_rig_BAS, "Parametre BAS, utilise avec P pour calculer la fractionnalite du greement."),
        with_dimension_hint(auto_rig_fractionality, "Fraction du greement, sans unite."),
        with_dimension_hint(auto_hull_Ff, "Franc-bord a l'avant."),
        with_dimension_hint(auto_hull_Fa, "Franc-bord a l'arriere."),
        with_dimension_hint(auto_hull_Boa, "Largeur maximale de la coque."),
        with_dimension_hint(auto_flat, "Facteur de creux de la voile, sans unite (1 = valeur nominale)."),
        auto_optimize_flat,
        with_dimension_hint(auto_flat_min, "Borne inferieure autorisee pour le facteur de creux."),
        with_dimension_hint(auto_reef, "Facteur de reduction de surface par ris, sans unite (1 = pleine surface)."),
        auto_optimize_reef,
        with_dimension_hint(auto_reef_min, "Borne inferieure autorisee pour le facteur de ris."),
        auto_pedro_hydroheel,
    ], layout=widgets.Layout(display="none"))
    def toggle_auto_pedro_controls(change: dict[str, Any]) -> None:
        auto_pedro_controls.layout.display = "" if is_pedro_aero_model(change["new"]) else "none"
    auto_aero_model.observe(toggle_auto_pedro_controls, names="value")
    def apply_auto_aero_configuration(state: BoatGeometryState) -> None:
        model = str(auto_aero_model.value)
        sail_source = AERO_MODEL_TO_SAIL_DATA[model]
        state.settings.aero_model = model
        state.settings.sail_coefficient_data_source = sail_source
        state.settings.pedro_hardcoded_hydroheel = bool(auto_pedro_hydroheel.value)
        if not is_pedro_aero_model(model):
            if hasattr(state, "manual_input"):
                delattr(state, "manual_input")
            return
        manual_input = clone_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        manual_input.solver.aero_model = model
        manual_input.solver.sail_coefficient_data_source = sail_source
        manual_input.main_sail.P_m = float(auto_main_sail_P.value)
        manual_input.main_sail.E_m = float(auto_main_sail_E.value)
        manual_input.main_sail.BAD_m = float(auto_main_sail_BAD.value)
        manual_input.sailset.sailset = str(auto_sailset.value)
        manual_input.sailset.spinnaker_type = str(auto_spinnaker_type.value)
        manual_input.sailset.reference_area_mode = str(auto_reference_area_mode.value)
        manual_input.sailset.flat = float(auto_flat.value)
        manual_input.sailset.reef = float(auto_reef.value)
        depowering_required = str(auto_sailset.value) in {"main_jib", "main_spinnaker"}
        manual_input.sailset.optimize_flat = bool(auto_optimize_flat.value or depowering_required)
        manual_input.sailset.optimize_reef = bool(auto_optimize_reef.value or depowering_required)
        manual_input.sailset.optimize_flat_reef = False
        manual_input.sailset.flat_min = float(auto_flat_min.value)
        manual_input.sailset.reef_min = float(auto_reef_min.value)
        manual_input.rig.I_m = float(auto_rig_I.value)
        manual_input.rig.J_m = float(auto_rig_J.value)
        manual_input.rig.LPG_m = float(auto_rig_LPG.value)
        manual_input.rig.SL_m = float(auto_rig_SL.value)
        manual_input.rig.EHM_m = float(auto_rig_EHM.value) if float(auto_rig_EHM.value) > 0 else None
        manual_input.rig.EMDC_m = float(auto_rig_EMDC.value) if float(auto_rig_EMDC.value) > 0 else None
        manual_input.rig.HBI_m = float(auto_rig_HBI.value)
        manual_input.rig.BAS_m = float(auto_rig_BAS.value) if float(auto_rig_BAS.value) > 0 else None
        manual_input.rig.fractionality = float(auto_rig_fractionality.value) if float(auto_rig_fractionality.value) > 0 else None
        manual_input.yacht.Ff_m = float(auto_hull_Ff.value)
        manual_input.yacht.Fa_m = float(auto_hull_Fa.value)
        manual_input.yacht.Boa_m = float(auto_hull_Boa.value)
        manual_input.advanced.aero_model = model
        manual_input.advanced.pedro_hardcoded_hydroheel = bool(auto_pedro_hydroheel.value)
        state.manual_input = manual_input
    use_rmv = widgets.Checkbox(value=True, description="Inclure la correction dynamique RMV")
    surface_mode = widgets.Dropdown(options=["Delft", "STL direct"], value="Delft", description="Surface")
    force_tol = widgets.FloatText(value=0.5, description="tol F N")
    moment_tol = widgets.FloatText(value=0.5, description="tol M Nm")
    max_nfev = widgets.IntText(value=180, description="max eval")

    analyze_button = widgets.Button(description="Analyser la geometrie", button_style="primary")
    quick_analyze_button = widgets.Button(description="Analyse rapide", button_style="warning", icon="bolt")
    fast_analysis_active = False
    solve_button = widgets.Button(description="Calculer le VPP", button_style="success", disabled=True)
    export_button = widgets.Button(description="Exporter CSV", button_style="")
    out_geom = widgets.Output()
    out_results = widgets.Output()
    out_diag = widgets.Output()
    vpp_progress = widgets.IntProgress(value=0, min=0, max=1, description="Calcul", bar_style="", layout=widgets.Layout(width="90%", display="none"))
    vpp_progress_status = widgets.HTML(value="", layout=widgets.Layout(width="90%"))
    cache_path = widgets.Text(value=str(Path("Résultats geometrie") / "ARGO_VPP_geometry_cache.txt"), description="Fichier TXT", layout=widgets.Layout(width="95%"))
    save_cache_button = widgets.Button(description="Enregistrer TXT", icon="save")
    load_cache_button = widgets.Button(description="Lire TXT", icon="folder-open")
    cache_status = widgets.HTML(value="")

    def refresh_stl_options(_button: widgets.Button | None = None) -> None:
        previous = stl_path.value
        options = available_stl_files()
        if not options:
            options = [GeometryInputs().stl_path]
        stl_path.options = options
        stl_path.value = previous if previous in options else options[0]

    def refresh_profile_options(_button: widgets.Button | None = None) -> None:
        options = available_naca_profiles(polar_folder.value)
        if not options:
            options = [("NACA0012 - profil manquant", "NACA0012")]
        values = [value for _label, value in options]
        dropdowns = [(wing_naca, ["NACA0012"]), (cb_naca, ["NACA0010", "NACA0012"]), (rudder_naca, ["NACA0009", "NACA0010", "NACA0008"])]
        for dropdown, preferred_codes in dropdowns:
            previous = dropdown.value
            dropdown.options = options
            preferred = next((code for code in preferred_codes if code in values), values[0])
            dropdown.value = previous if previous in values else preferred

    def collect_inputs() -> tuple[GeometryInputs, SolverSettings]:
        inputs = GeometryInputs(
            stl_path=stl_path.value,
            stl_unit=stl_unit.value,
            longitudinal_orientation=orientation.value,
            hull_draft_m=float(draft.value),
            wing_contour_text=wing_contour.value,
            centerboard_contour_text=cb_contour.value,
            rudder_contour_text=rudder_contour.value,
            wing_naca=wing_naca.value,
            centerboard_naca=cb_naca.value,
            rudder_naca=rudder_naca.value,
            hull_skin_thickness_mm=float(hull_t.value),
            centerboard_skin_thickness_mm=float(cb_t.value),
            rudder_skin_thickness_mm=float(rudder_t.value),
            wing_skin_thickness_mm=float(wing_t.value),
            equipment_mass_kg=float(equip_mass.value),
            equipment_x_m=float(equip_x.value),
            equipment_y_m=float(equip_y.value),
            equipment_z_m=float(equip_z.value),
            ballast_mode=ballast_mode.value,
            manual_ballast_kg=float(manual_ballast.value),
            fixed_alpha_deg=float(alpha.value),
            wing_alpha_mode=alpha_mode.value,
            alpha_min_deg=float(alpha_min.value),
            alpha_max_deg=float(alpha_max.value),
            alpha_step_deg=float(alpha_step.value),
            oswald_efficiency=float(oswald.value),
            delft_table_folder=delft_folder.value,
            polar_folder=polar_folder.value,
        )
        settings = SolverSettings(
            force_tol_N=float(force_tol.value),
            moment_tol_Nm=float(moment_tol.value),
            max_nfev=int(max_nfev.value),
            use_rmv=bool(use_rmv.value),
            wetted_surface_mode=surface_mode.value,
            aero_model=str(auto_aero_model.value),
            sail_coefficient_data_source=AERO_MODEL_TO_SAIL_DATA[str(auto_aero_model.value)],
            pedro_hardcoded_hydroheel=bool(auto_pedro_hydroheel.value),
        )
        return inputs, settings

    def on_analyze(_button: widgets.Button) -> None:
        with out_geom:
            clear_output()
            print("Analyse geometrique en cours...")
            try:
                inputs, settings = collect_inputs()
                if fast_analysis_active:
                    settings = SolverSettings(**asdict(settings))
                    settings.section_count = min(settings.section_count, 50)
                    settings.waterline_tol = max(settings.waterline_tol, 1e-5)
                    print("Mode rapide: maillage de sections reduit et tolerance hydrostatique assouplie. La plage GZ globale reste appliquee.")
                state = analyze_geometry(inputs, settings)
                apply_auto_aero_configuration(state)
                APP_STATE["geometry"] = state
                APP_STATE["geometry_summary"] = geometry_summary_dataframe(state)
                solve_button.disabled = False
                print("Analyse geometrique valide.")
                print("Modele 2-DOF : l'angle et la vitesse de derive ne sont pas calcules.")
                print("La force hydrodynamique laterale est imposee egale a la force aerodynamique.")
                with pd.option_context("display.max_rows", None, "display.max_columns", None):
                    display(APP_STATE["geometry_summary"])
                display(state.masses.component_table)
                if state.hydro.delft_warnings:
                    print("AVERTISSEMENTS DELFT:")
                    for warning in state.hydro.delft_warnings:
                        print(" -", warning)
                plot_3d_control(state)
                plot_gz_curve(state)
            except (ValueError, FileNotFoundError, RuntimeError) as exc:
                solve_button.disabled = True
                APP_STATE["geometry"] = None
                print("Erreur analyse geometrique:", exc)

    def on_solve(_button: widgets.Button) -> None:
        with out_results:
            clear_output()
            state = APP_STATE.get("geometry")
            if state is None:
                print("Analyser la geometrie avant de lancer le VPP.")
                return
            apply_auto_aero_configuration(state)
            if (
                getattr(state.settings, "geometry_input_mode", "") == "stl"
                and not state.hydrostatic_geometry_valid
            ):
                vpp_progress_status.value = "Géométrie hydrostatique invalide; nouvelle analyse STL..."
                try:
                    refreshed = analyze_geometry(state.inputs, state.settings)
                except Exception as exc:
                    message = f"Réanalyse STL échouée: {type(exc).__name__}: {exc}"
                    vpp_progress_status.value = message
                    print(message)
                    return
                APP_STATE["geometry"] = refreshed
                apply_auto_aero_configuration(refreshed)
                APP_STATE["geometry_summary"] = geometry_summary_dataframe(refreshed)
                state = refreshed
                if not refreshed.hydrostatic_geometry_valid:
                    message = (
                        "Géométrie toujours invalide après réanalyse: "
                        f"union={refreshed.hydrostatic_boolean_union_used}, "
                        f"chevauchement AABB={refreshed.mesh_overlap_detected}, "
                        f"volume AABB={refreshed.estimated_overlap_volume_m3:.6g} m3."
                    )
                    vpp_progress_status.value = message
                    print(message)
                    return
                vpp_progress_status.value = "Nouvelle analyse STL valide; lancement du VPP..."
            lo, hi = tws_range.value
            excluded = set(exclude_tws.value)
            tws = [k for k in range(lo, hi + 1) if k not in excluded]
            print(f"Calcul VPP pour TWS={tws} kn, TWA {TWA_RANGE_MIN_DEG:g}..{TWA_RANGE_MAX_DEG:g} deg en {TWA_RANGE_STEPS} steps.")
            vpp_progress.value = 0
            vpp_progress.max = 1
            vpp_progress.bar_style = ""
            vpp_progress.layout.display = ""
            vpp_progress_status.value = "Preparation du calcul..."

            def report_progress(completed: int, total: int, tws_value: float, twa_value: float, success: bool) -> None:
                vpp_progress.max = max(total, 1)
                vpp_progress.value = completed
                percent = round(100 * completed / max(total, 1))
                point_status = "OK" if success else "invalide"
                vpp_progress_status.value = (
                    f"{percent}% - point {completed}/{total}; "
                    f"TWS {tws_value:g} kn, TWA {twa_value:g} deg - {point_status}"
                )

            APP_STATE["_vpp_progress_callback"] = report_progress
            try:
                df = solve_vpp_grid(tws, state, progress_callback=report_progress)
                APP_STATE["results"] = df
                display(df)
                plot_all_results(df, state)
                valid_count = int(df["solver_success"].fillna(False).sum())
                total = len(df)
                vpp_progress.max = max(total, 1)
                vpp_progress.value = total
                vpp_progress.bar_style = "success" if valid_count == total else "warning"
                vpp_progress_status.value = f"Termine - {valid_count}/{total} points valides."
            except Exception as exc:
                vpp_progress.bar_style = "danger"
                vpp_progress_status.value = f"Calcul interrompu: {type(exc).__name__}: {exc}"
                raise
            finally:
                APP_STATE.pop("_vpp_progress_callback", None)

    def on_export(_button: widgets.Button) -> None:
        with out_diag:
            clear_output()
            if APP_STATE.get("results") is None or APP_STATE.get("geometry") is None:
                print("Aucun resultat a exporter.")
                return
            export_results(APP_STATE["results"], APP_STATE["geometry"])
            print("Exports crees: Résultats VPP/ARGO_VPP_results.csv, Résultats geometrie/ARGO_VPP_geometry.csv, Résultats geometrie/ARGO_VPP_GZ_curve.csv")

    def on_quick_analyze(_button: widgets.Button) -> None:
        nonlocal fast_analysis_active
        fast_analysis_active = True
        try:
            analyze_button.click()
        finally:
            fast_analysis_active = False

    def save_current_geometry() -> Path:
        state = APP_STATE.get("geometry")
        if state is None:
            raise ValueError("Analysez la geometrie avant d'enregistrer ses resultats.")
        return save_geometry_cache(state, cache_path.value)

    def on_save_cache(_button: widgets.Button) -> None:
        try:
            saved_path = save_current_geometry()
            cache_path.value = str(saved_path)
            cache_status.value = f"<span style='color:green'>Resultats enregistres: {saved_path}</span>"
        except Exception as exc:
            cache_status.value = f"<span style='color:red'>Erreur d'enregistrement: {type(exc).__name__}: {exc}</span>"

    def on_load_cache(_button: widgets.Button) -> None:
        try:
            state = load_geometry_cache(cache_path.value)
            APP_STATE["geometry"] = state
            APP_STATE["geometry_summary"] = geometry_summary_dataframe(state)
            apply_auto_aero_configuration(state)
            APP_STATE["results"] = None
            solve_button.disabled = False
            with out_geom:
                clear_output()
                print(f"Resultats lus depuis {cache_path.value}.")
                with pd.option_context("display.max_rows", None, "display.max_columns", None):
                    display(APP_STATE["geometry_summary"])
                display(state.masses.component_table)
                if state.hydro.delft_warnings:
                    print("AVERTISSEMENTS DELFT:")
                    for warning in state.hydro.delft_warnings:
                        print(" -", warning)
                plot_3d_control(state)
                plot_gz_curve(state)
            cache_status.value = f"<span style='color:green'>Geometrie et affichage 3D restaures depuis {cache_path.value}</span>"
        except Exception as exc:
            cache_status.value = f"<span style='color:red'>Erreur de lecture: {type(exc).__name__}: {exc}</span>"

    def on_analysis_finished(_button: widgets.Button) -> None:
        if APP_STATE.get("geometry") is None:
            return
        try:
            saved_path = save_current_geometry()
            cache_path.value = str(saved_path)
            cache_status.value = f"<span style='color:green'>Resultats enregistres: {saved_path}</span>"
        except Exception as exc:
            cache_status.value = f"<span style='color:darkorange'>Analyse terminee, mais cache non enregistre: {exc}</span>"

    analyze_button.on_click(on_analyze)
    quick_analyze_button.on_click(on_quick_analyze)
    save_cache_button.on_click(on_save_cache)
    load_cache_button.on_click(on_load_cache)
    analyze_button.on_click(on_analysis_finished)
    solve_button.on_click(on_solve)
    export_button.on_click(on_export)
    refresh_stl.on_click(refresh_stl_options)
    refresh_profiles.on_click(refresh_profile_options)

    tabs = widgets.Tab(children=[
        widgets.VBox([widgets.HTML("<b>Modele 2-DOF : derive non calculee, force laterale imposee.</b>"), with_dimension_hint(tws_range, "Plage de vitesse du vent vrai (TWS), en noeuds."), exclude_tws]),
        widgets.VBox([widgets.HBox([stl_path, refresh_stl]), stl_unit, orientation, with_dimension_hint(draft, "Tirant d'eau de la coque sous la ligne de flottaison."), delft_folder, widgets.HBox([analyze_button, quick_analyze_button])]),
        widgets.VBox([widgets.HBox([polar_folder, refresh_profiles]), wing_contour, wing_naca, with_dimension_hint(wing_t, "Epaisseur de l'aile / grand-voile, en millimetres."), alpha_mode, with_dimension_hint(alpha, "Angle d'attaque impose de l'aile."), with_dimension_hint(alpha_min, "Angle d'attaque minimal explore par le solveur."), with_dimension_hint(alpha_max, "Angle d'attaque maximal explore par le solveur."), with_dimension_hint(alpha_step, "Pas angulaire du balayage de l'angle d'attaque."), with_dimension_hint(oswald, "Facteur d'efficacite d'Oswald, sans unite."), auto_aero_model, auto_pedro_controls]),
        widgets.VBox([cb_contour, cb_naca, with_dimension_hint(cb_t, "Epaisseur de la derive, en millimetres.")]),
        widgets.VBox([rudder_contour, rudder_naca, with_dimension_hint(rudder_t, "Epaisseur du safran, en millimetres.")]),
        widgets.VBox([ballast_mode, with_dimension_hint(manual_ballast, "Masse de ballast ajoutee."), with_dimension_hint(equip_mass, "Masse des equipements embarques."), widgets.HBox([with_dimension_hint(equip_x, "Position longitudinale du centre de masse des equipements."), with_dimension_hint(equip_y, "Position laterale du centre de masse des equipements."), with_dimension_hint(equip_z, "Hauteur du centre de masse des equipements.")])]),
        widgets.VBox([use_rmv, surface_mode, with_dimension_hint(force_tol, "Tolerance sur le residu de force."), with_dimension_hint(moment_tol, "Tolerance sur le residu de moment."), with_dimension_hint(max_nfev, "Nombre maximal d'evaluations du solveur.")]),
        widgets.VBox([widgets.HTML("<b>Cache des resultats d'analyse geometrique</b>"), cache_path, widgets.HBox([save_cache_button, load_cache_button]), cache_status, out_geom]),
        widgets.VBox([solve_button, vpp_progress, vpp_progress_status, out_results]),
        widgets.VBox([export_button, out_diag]),
    ])
    for i, title in enumerate([
        "1. Environnement", "2. Coque STL", "3. Aile", "4. Derive", "5. Safran",
        "6. Masses et ballast", "7. Parametres numeriques", "8. Geometrie calculee",
        "9. Resultats", "10. Diagnostics"
    ]):
        tabs.set_title(i, title)
    return tabs


ui = build_widgets()
display(ui)

# BEGIN FINAL_GEOMETRY_OVERLAP_DIAGNOSTICS
def estimate_aabb_overlap_volume(mesh_a: trimesh.Trimesh, mesh_b: trimesh.Trimesh) -> float:
    """Conservative axis-aligned bounding-box overlap estimate in m3."""

    a0, a1 = np.asarray(mesh_a.bounds[0], dtype=float), np.asarray(mesh_a.bounds[1], dtype=float)
    b0, b1 = np.asarray(mesh_b.bounds[0], dtype=float), np.asarray(mesh_b.bounds[1], dtype=float)
    lo = np.maximum(a0, b0)
    hi = np.minimum(a1, b1)
    ext = np.maximum(hi - lo, 0.0)
    return float(np.prod(ext))


def hydrostatic_mesh_union_or_concat(meshes: list[trimesh.Trimesh]) -> tuple[trimesh.Trimesh, bool]:
    """Try a boolean union when an engine is advertised; otherwise concatenate with diagnostics."""

    engines = getattr(getattr(trimesh, "boolean", None), "engines_available", set())
    if engines:
        try:
            union = trimesh.boolean.union(meshes)
            if isinstance(union, trimesh.Trimesh) and len(union.faces):
                return union, True
        except BaseException as exc:
            warnings.warn(f"Union booleenne hydrostatique indisponible ou echouee: {exc}")
    return combine_meshes(meshes), False


def hydrostatic_overlap_diagnostics(meshes: list[trimesh.Trimesh]) -> tuple[bool, float]:
    """Detect possible overlaps between hydrostatic meshes without pretending exact union volume."""

    estimate = 0.0
    for i in range(len(meshes)):
        for j in range(i + 1, len(meshes)):
            estimate += estimate_aabb_overlap_volume(meshes[i], meshes[j])
    return bool(estimate > 1e-10), float(estimate)


def analyze_geometry(inputs: GeometryInputs, settings: SolverSettings) -> BoatGeometryState:
    """Run geometry pipeline; wing is excluded from hydrostatic geometry and overlap is diagnosed."""

    print("1/8 Lecture des tables Delft...")
    tables = load_delft_tables(inputs.delft_table_folder)
    print("2/8 Chargement et controle du STL...")
    hull = load_hull_stl(inputs.stl_path, inputs.stl_unit, inputs.longitudinal_orientation)
    diagnostics = validate_mesh(hull)
    blocking_errors = [err for err in diagnostics["errors"] if not err.startswith("Maillage non etanche")]
    if blocking_errors:
        raise ValueError("STL defectueux: " + "; ".join(blocking_errors))
    if not diagnostics["is_watertight"]:
        print("AVERTISSEMENT: STL complet non etanche. Le calcul continue et verifiera le volume immerge ferme par le plan d'eau.")
    print("3/8 Decoupage au plan d'eau et hydrostatique...")
    hydro, immersed_hull, loops, wet_triangles = compute_hull_hydrostatics(hull, inputs.hull_draft_m, tables, settings.section_count)
    print(f"    Volume immerge coque: {hydro.volume_C:.5f} m3, A_WP: {hydro.A_WP:.5f} m2")
    positions = automatic_positions(hull, hydro)
    print("4/8 Generation aile, derive et safran...")
    wing = build_foil_mesh_from_contour("aile", parse_contour_text(inputs.wing_contour_text, "aile"), inputs.wing_naca, positions["wing"], "up")
    centerboard = build_foil_mesh_from_contour("derive", parse_contour_text(inputs.centerboard_contour_text, "derive"), inputs.centerboard_naca, positions["centerboard"], "down")
    rudder = build_foil_mesh_from_contour("safran", parse_contour_text(inputs.rudder_contour_text, "safran"), inputs.rudder_naca, positions["rudder"], "down")

    print("5/8 Calcul des masses et du ballast...")
    hydro_meshes = [hull, centerboard.mesh, rudder.mesh]
    mesh_overlap_detected, estimated_overlap_volume = hydrostatic_overlap_diagnostics(hydro_meshes)
    if mesh_overlap_detected:
        print(f"AVERTISSEMENT: chevauchement hydrostatique possible estime par AABB: {estimated_overlap_volume:.6g} m3.")
    buoyancy_mesh, union_used = hydrostatic_mesh_union_or_concat(hydro_meshes)
    if union_used:
        total_displaced_volume = safe_mesh_volume(buoyancy_mesh)
    else:
        total_displaced_volume = hydro.volume_C + centerboard.volume_m3 + rudder.volume_m3
    hydrostatic_geometry_valid = bool(union_used or not mesh_overlap_detected)
    T_max, z_min_submerged = compute_true_T_max(hydro_meshes, hydro.waterline_z)
    hull_area, hull_surface_centroid = surface_area_centroid(hull.triangles)
    ballast_position = np.array([centerboard.volume_centroid[0], 0.0, float(centerboard.mesh.vertices[:, 2].min())])
    masses = compute_mass_properties(inputs, hull_area, hull_surface_centroid, wing, centerboard, rudder, total_displaced_volume, ballast_position)
    print("6/8 Surface laterale projetee...")
    lateral_area = compute_lateral_projected_area(hydro_meshes, hydro.waterline_z)
    print(f"7/8 Courbe GZ {HEEL_RANGE_MIN_DEG:g}-{HEEL_RANGE_MAX_DEG:g} deg...")
    gz_curve = compute_gz_curve(
        buoyancy_mesh,
        masses,
        total_displaced_volume,
        min_angle_deg=HEEL_RANGE_MIN_DEG,
        max_angle_deg=HEEL_RANGE_MAX_DEG,
        angle_count=HEEL_RANGE_STEPS + 1,
        waterline_tol=settings.waterline_tol,
    )
    print("8/8 Synthese geometrique...")
    return BoatGeometryState(
        inputs=inputs,
        settings=settings,
        hull_mesh=hull,
        immersed_hull_mesh=immersed_hull,
        hydro=hydro,
        wing=wing,
        centerboard=centerboard,
        rudder=rudder,
        masses=masses,
        total_displaced_volume_m3=total_displaced_volume,
        T_max_m=T_max,
        z_min_submerged_m=z_min_submerged,
        total_draft_m=T_max,
        lateral_area_m2=lateral_area,
        gz_curve=gz_curve,
        delft_tables=tables,
        hydrostatic_boolean_union_used=union_used,
        mesh_overlap_detected=mesh_overlap_detected,
        estimated_overlap_volume_m3=estimated_overlap_volume,
        hydrostatic_geometry_valid=hydrostatic_geometry_valid,
        wetted_surface_cache={},
    )


def geometry_summary_dataframe(state: BoatGeometryState) -> pd.DataFrame:
    hydro = asdict(state.hydro)
    hydro.pop("delft_warnings", None)
    rows = [{"group": "coque", "parameter": key, "value": value} for key, value in hydro.items()]
    for foil in [state.wing, state.centerboard, state.rudder]:
        for key in ["planform_area_m2", "external_area_m2", "wetted_area_m2", "span_m", "mean_chord_m", "volume_m3", "aspect_ratio", "thickness_ratio"]:
            rows.append({"group": foil.name, "parameter": key, "value": getattr(foil, key)})
        rows.append({"group": foil.name, "parameter": "root_x_m", "value": foil.root_position[0]})
        rows.append({"group": foil.name, "parameter": "root_y_m", "value": foil.root_position[1]})
        rows.append({"group": foil.name, "parameter": "root_z_m", "value": foil.root_position[2]})
    rows.extend([
        {"group": "masses", "parameter": "total_mass_kg", "value": state.masses.total_mass_kg},
        {"group": "masses", "parameter": "ballast_mass_kg", "value": state.masses.ballast_mass_kg},
        {"group": "masses", "parameter": "CG_x_m", "value": state.masses.center_of_gravity[0]},
        {"group": "masses", "parameter": "CG_y_m", "value": state.masses.center_of_gravity[1]},
        {"group": "masses", "parameter": "CG_z_m", "value": state.masses.center_of_gravity[2]},
        {"group": "hydro", "parameter": "total_displaced_volume_m3", "value": state.total_displaced_volume_m3},
        {"group": "hydro", "parameter": "T_max_m", "value": state.T_max_m},
        {"group": "hydro", "parameter": "z_min_submerged_m", "value": state.z_min_submerged_m},
        {"group": "hydro", "parameter": "lateral_area_m2", "value": state.lateral_area_m2},
        {"group": "hydro", "parameter": "hydrostatic_boolean_union_used", "value": state.hydrostatic_boolean_union_used},
        {"group": "hydro", "parameter": "mesh_overlap_detected", "value": state.mesh_overlap_detected},
        {"group": "hydro", "parameter": "estimated_overlap_volume_m3", "value": state.estimated_overlap_volume_m3},
        {"group": "hydro", "parameter": "hydrostatic_geometry_valid", "value": state.hydrostatic_geometry_valid},
    ])
    return pd.DataFrame(rows)
# END FINAL_GEOMETRY_OVERLAP_DIAGNOSTICS


In [20]:
def automatic_positions(mesh: trimesh.Trimesh, hydro: HullHydrostatics) -> dict[str, np.ndarray]:
    """Position the appendage roots with a small controlled insertion into the hull."""

    bounds = mesh.bounds
    length = max(float(mesh.extents[0]), 1e-6)
    x_mid = hydro.LCB if np.isfinite(hydro.LCB) else 0.5 * (bounds[0, 0] + bounds[1, 0])
    x_rudder = bounds[0, 0] + 0.08 * length
    attachment_inset = float(np.clip(0.01 * max(float(mesh.extents[2]), 1e-6), 0.003, 0.02))
    centerboard_root_z = hull_z_at_x(mesh, x_mid, top=False) + attachment_inset
    rudder_root_z = hull_z_at_x(mesh, x_rudder, top=False) + attachment_inset
    return {
        "wing": np.array([x_mid, 0.0, hull_z_at_x(mesh, x_mid, top=True)]),
        "centerboard": np.array([x_mid, 0.0, centerboard_root_z]),
        "rudder": np.array([x_rudder, 0.0, rudder_root_z]),
    } 


def hydrostatic_mesh_union_or_concat(meshes: list[trimesh.Trimesh]) -> tuple[trimesh.Trimesh, bool]:
    """Orient closed solids outward before their hydrostatic boolean union."""

    engines = getattr(getattr(trimesh, "boolean", None), "engines_available", set())
    engine = "manifold" if "manifold" in engines else None
    union_inputs: list[trimesh.Trimesh] = []
    corrected_orientation = False
    for mesh in meshes:
        candidate = mesh.copy()
        if candidate.is_watertight and not candidate.is_winding_consistent:
            candidate.fix_normals()
        if candidate.is_watertight and np.isfinite(candidate.volume) and candidate.volume < 0.0:
            candidate.invert()
            corrected_orientation = True
        if not candidate.is_volume:
            warnings.warn("Union hydrostatique impossible: un maillage n'est pas un volume ferme a normales sortantes.")
            return combine_meshes(meshes), False
        union_inputs.append(candidate)
    if corrected_orientation:
        warnings.warn("Orientation interne detectee: normales inversees sur les copies utilisees pour l'union hydrostatique.")
    try:
        union = trimesh.boolean.union(union_inputs, engine=engine) if engine is not None else trimesh.boolean.union(union_inputs)
        if (
            isinstance(union, trimesh.Trimesh)
            and len(union.faces) > 0
            and union.is_volume
            and np.isfinite(union.volume)
            and union.volume > 1e-12
        ):
            return union, True
        warnings.warn("Union hydrostatique invalide: maillage non ferme ou volume non positif.")
    except Exception as exc:
        backend = engine or "automatique"
        warnings.warn(f"Union hydrostatique ({backend}) echouee: {exc}")
    return combine_meshes(meshes), False

In [21]:
def analyze_geometry(inputs: GeometryInputs, settings: SolverSettings) -> BoatGeometryState:
    """Analyze STL geometry using the submerged union volume for displacement and GZ."""

    print("1/8 Lecture des tables Delft...")
    tables = load_delft_tables(inputs.delft_table_folder)
    print("2/8 Chargement et controle du STL...")
    hull = load_hull_stl(inputs.stl_path, inputs.stl_unit, inputs.longitudinal_orientation)
    diagnostics = validate_mesh(hull)
    blocking_errors = [err for err in diagnostics["errors"] if not err.startswith("Maillage non etanche")]
    if blocking_errors:
        raise ValueError("STL defectueux: " + "; ".join(blocking_errors))
    if not diagnostics["is_watertight"]:
        print("AVERTISSEMENT: STL complet non etanche. Le calcul continue et verifiera le volume immerge ferme par le plan d'eau.")
    print("3/8 Decoupage au plan d'eau et hydrostatique...")
    hydro, immersed_hull, loops, wet_triangles = compute_hull_hydrostatics(
        hull, inputs.hull_draft_m, tables, settings.section_count
    )
    print(f"    Volume immerge coque: {hydro.volume_C:.5f} m3, A_WP: {hydro.A_WP:.5f} m2")
    positions = automatic_positions(hull, hydro)
    print("4/8 Generation aile, derive et safran...")
    wing = build_foil_mesh_from_contour(
        "aile", parse_contour_text(inputs.wing_contour_text, "aile"),
        inputs.wing_naca, positions["wing"], "up",
    )
    centerboard = build_foil_mesh_from_contour(
        "derive", parse_contour_text(inputs.centerboard_contour_text, "derive"),
        inputs.centerboard_naca, positions["centerboard"], "down",
    )
    rudder = build_foil_mesh_from_contour(
        "safran", parse_contour_text(inputs.rudder_contour_text, "safran"),
        inputs.rudder_naca, positions["rudder"], "down",
    )

    print("5/8 Calcul des masses et du ballast...")
    hydro_meshes = [hull, centerboard.mesh, rudder.mesh]
    mesh_overlap_detected, estimated_overlap_volume = hydrostatic_overlap_diagnostics(hydro_meshes)
    if mesh_overlap_detected:
        print(f"AVERTISSEMENT: chevauchement hydrostatique possible estime par AABB: {estimated_overlap_volume:.6g} m3.")
    buoyancy_mesh, union_used = hydrostatic_mesh_union_or_concat(hydro_meshes)
    if union_used:
        immersed_union, _, _ = immersed_mesh_for_waterline(
            buoyancy_mesh, hydro.waterline_z, cap=True
        )
        total_displaced_volume = safe_mesh_volume(immersed_union)
        if not np.isfinite(total_displaced_volume) or total_displaced_volume <= 0.0:
            raise ValueError("Volume immerge de l'union hydrostatique nul ou invalide.")
    else:
        appendage_volumes = [
            safe_mesh_volume(immersed_mesh_for_waterline(foil, hydro.waterline_z, cap=True)[0])
            for foil in [centerboard.mesh, rudder.mesh]
        ]
        total_displaced_volume = float(hydro.volume_C + sum(appendage_volumes))
    hydrostatic_geometry_valid = bool(union_used or not mesh_overlap_detected)
    T_max, z_min_submerged = compute_true_T_max(hydro_meshes, hydro.waterline_z)
    hull_area, hull_surface_centroid = surface_area_centroid(hull.triangles)
    ballast_position = np.array([
        centerboard.volume_centroid[0],
        0.0,
        float(centerboard.mesh.vertices[:, 2].min()),
    ])
    masses = compute_mass_properties(
        inputs, hull_area, hull_surface_centroid, wing, centerboard, rudder,
        total_displaced_volume, ballast_position,
    )
    print("6/8 Surface laterale projetee...")
    lateral_area = compute_lateral_projected_area(hydro_meshes, hydro.waterline_z)
    print(f"7/8 Courbe GZ {HEEL_RANGE_MIN_DEG:g}-{HEEL_RANGE_MAX_DEG:g} deg...")
    gz_curve = compute_gz_curve(
        buoyancy_mesh,
        masses,
        total_displaced_volume,
        min_angle_deg=HEEL_RANGE_MIN_DEG,
        max_angle_deg=HEEL_RANGE_MAX_DEG,
        angle_count=HEEL_RANGE_STEPS + 1,
        waterline_tol=settings.waterline_tol,
    )
    print("8/8 Synthese geometrique...")
    return BoatGeometryState(
        inputs=inputs,
        settings=settings,
        hull_mesh=hull,
        immersed_hull_mesh=immersed_hull,
        hydro=hydro,
        wing=wing,
        centerboard=centerboard,
        rudder=rudder,
        masses=masses,
        total_displaced_volume_m3=total_displaced_volume,
        T_max_m=T_max,
        z_min_submerged_m=z_min_submerged,
        total_draft_m=T_max,
        lateral_area_m2=lateral_area,
        gz_curve=gz_curve,
        delft_tables=tables,
        hydrostatic_boolean_union_used=union_used,
        mesh_overlap_detected=mesh_overlap_detected,
        estimated_overlap_volume_m3=estimated_overlap_volume,
        hydrostatic_geometry_valid=hydrostatic_geometry_valid,
        wetted_surface_cache={},
    )

In [23]:
_attachment_test_hull = trimesh.creation.box(extents=[4.0, 2.0, 2.0])
_attachment_test_hull.apply_translation([2.0, 0.0, -0.5])
_attachment_test_hull.invert()
_attachment_test_hydro = HullHydrostatics(
    LWL=4.0, BWL=2.0, TC=1.5, volume_C=6.0, S_C=20.0, A_WP=8.0,
    LCB=2.0, LCF=2.0, AMS_CB=2.0, C_M=1.0, C_P=1.0, x_FP=4.0,
    LCB_fpp=2.0, LCF_fpp=2.0, LCB_pct_midship=0.0, LCF_pct_midship=0.0,
    waterline_z=0.0, B_CB=2.0, delft_valid=True,
)
_attachment_test_positions = automatic_positions(_attachment_test_hull, _attachment_test_hydro)
_attachment_test_inset = _attachment_test_positions["rudder"][2] - _attachment_test_hull.bounds[0, 2]
assert 0.003 <= _attachment_test_inset <= 0.020001, _attachment_test_inset
_attachment_test_foil = trimesh.creation.box(extents=[0.3, 0.1, 1.0])
_attachment_test_foil.apply_translation([
    _attachment_test_positions["rudder"][0],
    0.0,
    _attachment_test_positions["rudder"][2] - 0.5,
])
_attachment_test_foil.invert()
_attachment_test_overlap, _attachment_test_overlap_volume = hydrostatic_overlap_diagnostics([
    _attachment_test_hull, _attachment_test_foil,
])
_attachment_test_union, _attachment_test_union_used = hydrostatic_mesh_union_or_concat([
    _attachment_test_hull, _attachment_test_foil,
])
assert _attachment_test_overlap and _attachment_test_overlap_volume > 0.0
assert _attachment_test_union_used and _attachment_test_union.is_volume
_attachment_test_immersed_union, _, _ = immersed_mesh_for_waterline(
    _attachment_test_union, _attachment_test_hydro.waterline_z, cap=True,
)
_attachment_test_displacement = safe_mesh_volume(_attachment_test_immersed_union)
assert 0.0 < _attachment_test_displacement < _attachment_test_union.volume
_attachment_test_z, _ = find_waterline_for_target_volume(
    _attachment_test_union,
    _attachment_test_displacement,
    (-2.6, 0.6),
    tol=1e-7,
)
assert abs(_attachment_test_z) < 1e-6, _attachment_test_z
print(
    f"Union hydrostatique valide: insertion={_attachment_test_inset:.3f} m; "
    f"volume total={_attachment_test_union.volume:.6f} m3; "
    f"volume immerge={_attachment_test_displacement:.6f} m3."
)

C:\Users\erik.grenache\AppData\Local\Temp\ipykernel_19352\796601544.py:37: UserWarning: Orientation interne detectee: normales inversees sur les copies utilisees pour l'union hydrostatique.
  warnings.warn("Orientation interne detectee: normales inversees sur les copies utilisees pour l'union hydrostatique.")
C:\Users\erik.grenache\AppData\Local\Temp\ipykernel_19352\796601544.py:51: UserWarning: Union hydrostatique (automatique) echouee: No boolean backend: `pip install manifold3d` or install `blender`
  warnings.warn(f"Union hydrostatique ({backend}) echouee: {exc}")


AssertionError: 

In [ ]:
# BEGIN MANUAL_INPUT_WIDGETS_DISPLAY
manual_ui = build_manual_input_widgets()
display(manual_ui)
# END MANUAL_INPUT_WIDGETS_DISPLAY

## 16. Graphiques Matplotlib

In [ ]:
def prepare_plot_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Copy results and expose NaN plot speeds for every invalid point."""

    out = df.copy()
    if "validity_flag" in out.columns:
        valid = out["validity_flag"].astype(str).str.lower().eq("valid")
    elif "solver_success" in out.columns:
        valid = out["solver_success"].fillna(False).astype(bool)
    else:
        valid = pd.Series(True, index=out.index)
    speed_col = "boat_speed_kn" if "boat_speed_kn" in out.columns else "boat_speed_mps"
    out["boat_speed_plot"] = np.where(valid, pd.to_numeric(out[speed_col], errors="coerce"), np.nan)
    return out


def mirror_polar(df: pd.DataFrame) -> pd.DataFrame:
    """Mirror half polar results to 0..360 degrees."""

    df = prepare_plot_dataframe(df)
    mirror = df.copy()
    mirror["TWA_deg"] = 360.0 - mirror["TWA_deg"]
    out = pd.concat([df, mirror], ignore_index=True)
    return out.drop_duplicates(subset=["TWS_kn", "TWA_deg"]).sort_values(["TWS_kn", "TWA_deg"])


def _valid_plot_group(group: pd.DataFrame, col: str) -> np.ndarray:
    values = group[col].to_numpy(dtype=float)
    if "validity_flag" in group.columns:
        valid = group["validity_flag"].astype(str).str.lower().eq("valid").to_numpy()
        values = np.where(valid, values, np.nan)
    elif "solver_success" in group.columns:
        values = np.where(group["solver_success"].fillna(False).to_numpy(dtype=bool), values, np.nan)
    return values


def _set_speed_ylim(ax: Any, values: list[np.ndarray]) -> None:
    """Scale speed charts from finite, non-negative valid boat speeds."""

    finite_values = [
        value[np.isfinite(value) & (value >= 0.0)]
        for value in (np.asarray(series, dtype=float) for series in values)
    ]
    finite_values = [value for value in finite_values if value.size]
    if not finite_values:
        return
    max_speed = float(np.max(np.concatenate(finite_values)))
    ax.set_ylim(0.0, max_speed * 1.1 if max_speed > 0.0 else 1.0)


def plot_speed_polar(df: pd.DataFrame) -> None:
    """Plot boat speed polar in knots without connecting invalid points."""

    data = mirror_polar(df)
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection="polar")
    speed_values = []
    for tws, group in data.groupby("TWS_kn"):
        theta = np.deg2rad(group["TWA_deg"].to_numpy())
        speed = _valid_plot_group(group, "boat_speed_kn")
        speed_values.append(speed)
        ax.plot(theta, speed, label=f"{tws:g} kn")
    _set_speed_ylim(ax, speed_values)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title("Polaire de vitesse - TWA cap du bateau par rapport au vent reel")
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
    plt.show()


def plot_lines_by_tws(df: pd.DataFrame, columns: list[str], title: str, ylabel: str) -> None:
    """Generic TWA line plot by TWS, with invalid solutions separated by NaN."""

    fig, ax = plt.subplots()
    speed_values = []
    for tws, group in df.groupby("TWS_kn"):
        for col in columns:
            if col in group:
                values = _valid_plot_group(group, col)
                ax.plot(group["TWA_deg"], values, label=f"{col} - {tws:g} kn")
                if col in {"boat_speed_kn", "boat_speed_mps"}:
                    speed_values.append(values)
    if speed_values:
        _set_speed_ylim(ax, speed_values)
    ax.set_xlabel("TWA - cap du bateau par rapport au vent reel [deg]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.show()


def plot_convergence_residuals(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots()
    iteration_ax = ax.twinx()
    for tws, group in df.groupby("TWS_kn"):
        for col in ["force_residual_N", "moment_residual_Nm"]:
            if col in group:
                ax.plot(group["TWA_deg"], _valid_plot_group(group, col), label=f"{col} - {tws:g} kn")
        if "solver_iterations" in group:
            iteration_ax.step(
                group["TWA_deg"],
                _valid_plot_group(group, "solver_iterations"),
                where="mid",
                marker="o",
                markersize=3,
                linestyle="--",
                label=f"Itérations solveur - {tws:g} kn",
            )
    ax.set_xlabel("TWA - cap du bateau par rapport au vent reel [deg]")
    ax.set_ylabel("Residus [N / N.m]")
    iteration_ax.set_ylabel("Nombre d'iterations")
    iteration_ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    iteration_ax.set_ylim(bottom=0)
    ax.set_title("Residus de convergence")
    handles, labels = ax.get_legend_handles_labels()
    iteration_handles, iteration_labels = iteration_ax.get_legend_handles_labels()
    ax.legend(handles + iteration_handles, labels + iteration_labels, fontsize=8)
    ax.grid(True, alpha=0.25)
    plt.show()


def plot_gz_curve(state: BoatGeometryState) -> None:
    """Plot signed GZ and hydrostatic righting moment."""

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(state.gz_curve["phi_deg"], state.gz_curve["GZ_signed_m"])
    ax[0].set_xlabel("gite [deg]")
    ax[0].set_ylabel("GZ signe [m]")
    ax[0].set_title("Courbe GZ signee")
    ax[1].plot(state.gz_curve["phi_deg"], state.gz_curve["M_hydrostatic_restoring_Nm"])
    ax[1].set_xlabel("gite [deg]")
    ax[1].set_ylabel("M hydrostatique disponible [N.m]")
    ax[1].set_title("Moment redressant hydrostatique")
    plt.tight_layout()
    plt.show()


def plot_all_results(df: pd.DataFrame, state: BoatGeometryState, include_gz: bool = True) -> None:
    """Create the requested Matplotlib diagnostics."""

    if df.empty:
        print("Aucun resultat a tracer.")
        return
    plot_speed_polar(df)
    plot_lines_by_tws(df, ["boat_speed_kn"], "Vitesse bateau", "V_B [kn]")
    plot_lines_by_tws(df, ["heel_deg"], "Angle de gite", "phi [deg]")
    plot_lines_by_tws(df, ["AWS_kn"], "Vitesse du vent apparent", "AWS [kn]")
    plot_lines_by_tws(df, ["AWA_deg"], "Angle du vent apparent", "AWA [deg]")
    plot_lines_by_tws(df, ["F_drive_N", "R_total_N", "force_residual_N"], "Equilibre longitudinal", "Force [N]")
    plot_lines_by_tws(df, ["F_side_aero_N", "F_side_hydro_assumed_N"], "Equilibre lateral impose - modele 2-DOF", "Force [N]")
    plot_lines_by_tws(df, ["M_heel_total_Nm", "M_right_total_Nm", "moment_residual_Nm"], "Equilibre des moments", "Moment [N.m]")
    plot_lines_by_tws(df, ["M_aero_about_WL_Nm", "M_hydroheel_Nm", "M_hydrostatic_Nm", "RMV_Nm"], "Contributions moments", "Moment [N.m]")
    resistance_cols = [
        "R_hull_viscous_N", "R_centerboard_viscous_N", "R_rudder_viscous_N",
        "R_hull_residuary_N", "R_centerboard_residuary_N", "R_heel_friction_delta_N",
        "R_heel_hull_residuary_delta_N", "R_heel_keel_residuary_delta_N", "R_induced_N",
        "R_rail_under_N", "R_total_N",
    ]
    plot_lines_by_tws(df, resistance_cols, "Decomposition de la resistance", "Resistance [N]")
    plot_lines_by_tws(df, ["Fn"], "Nombre de Froude", "Fn [-]")
    plot_lines_by_tws(df, ["Cl_wing", "Cd_wing"], "Coefficients aile", "CL / CD [-]")
    plot_lines_by_tws(df, ["Re_wing"], "Reynolds aile", "Re [-]")
    plot_lines_by_tws(df, ["alpha_wing_deg", "wing_orientation_deg"], "Incidence et orientation aile", "deg")
    plot_lines_by_tws(df, ["T_max_m", "T_eff_m"], "Tirants d'eau diagnostiques", "m")
    plot_convergence_residuals(df)
    if include_gz:
        plot_gz_curve(state)
    phi = np.arange(0, 61, 5)
    S_values = []
    label = "Surface mouillee STL direct" if state.settings.wetted_surface_mode == "STL direct" else "Surface mouillee Delft"
    for p in phi:
        try:
            if state.settings.wetted_surface_mode == "STL direct":
                S, _ = direct_hull_wetted_surface_delta(1.0, deg_to_rad(float(p)), state)
            else:
                S, _, _, _, _ = wetted_surface_heel_delta(1.0, deg_to_rad(float(p)), state.hydro, state.delft_tables)
        except TableDomainError:
            S = np.nan
        S_values.append(S)
    plt.figure()
    plt.plot(phi, S_values, label=label)
    plt.axhline(state.hydro.S_C, color="k", linestyle="--", label="Surface STL droite")
    plt.xlabel("gite [deg]")
    plt.ylabel("Surface [m2]")
    plt.title("Surface mouillee")
    plt.legend()
    plt.show()

In [ ]:
_speed_plot_test_df = pd.DataFrame({
    "TWS_kn": [6.0, 6.0, 6.0],
    "TWA_deg": [30.0, 90.0, 150.0],
    "boat_speed_kn": [2.0, 4.0, 50.0],
    "solver_success": [True, True, False],
})
_original_plot_show = plt.show
plt.show = lambda *args, **kwargs: None
try:
    plot_speed_polar(_speed_plot_test_df)
    _polar_speed_ylim = plt.gca().get_ylim()
    plt.close("all")
    plot_lines_by_tws(_speed_plot_test_df, ["boat_speed_kn"], "Vitesse bateau", "V_B [kn]")
    _line_speed_ylim = plt.gca().get_ylim()
    assert np.allclose(_polar_speed_ylim, (0.0, 4.4)), _polar_speed_ylim
    assert np.allclose(_line_speed_ylim, (0.0, 4.4)), _line_speed_ylim
finally:
    plt.show = _original_plot_show
    plt.close("all")
print(f"Echelles vitesse valides: polaire={_polar_speed_ylim}, courbe={_line_speed_ylim}; point invalide 50 kn exclu.")


Echelles vitesse valides: polaire=(np.float64(0.0), np.float64(4.4)), courbe=(np.float64(0.0), np.float64(4.4)); point invalide 50 kn exclu.


## 17. Export des resultats

In [ ]:
def export_results(results: pd.DataFrame, state: BoatGeometryState, folder: str | Path = ".") -> None:
    """Export VPP results and geometry-analysis CSV files."""

    root = Path(folder)
    root.mkdir(parents=True, exist_ok=True)
    vpp_folder = root / "Résultats VPP"
    vpp_folder.mkdir(parents=True, exist_ok=True)
    results.to_csv(vpp_folder / "ARGO_VPP_results.csv", index=False)
    geometry_folder = root / "Résultats geometrie"
    geometry_folder.mkdir(parents=True, exist_ok=True)
    geometry_summary_dataframe(state).to_csv(geometry_folder / "ARGO_VPP_geometry.csv", index=False)
    state.gz_curve.to_csv(geometry_folder / "ARGO_VPP_GZ_curve.csv", index=False)

## Mode STL et mode entree manuelle

`geometry_input_mode = "stl"` conserve le flux existant: lecture du STL, decoupe au plan d'eau et extraction automatique des grandeurs hydrostatiques.

`geometry_input_mode = "manual"` construit un `BoatGeometryState` compatible avec `solve_vpp_grid()` depuis des parametres scalaires. Ce mode sert a comparer rapidement ARGO-VPP avec Python-VPP ou yacht-vpp.streamlit.app. Sa precision depend directement de la qualite des entrees fournies.

Le modele reste un VPP stationnaire 2-DOF: equilibre longitudinal et equilibre en roulis. Le mode `orc_simplified` est un mode ORC simplifie, pas une reproduction complete de l'ORC 2026.

## Correspondance avec les champs Python-VPP

| Python-VPP / site | ARGO-VPP manual input | Unite | Remarque |
|---|---|---|---|
| Lwl | Lwl_m | m | Longueur a la flottaison |
| âˆ‡ | displaced_volume_m3 | m3 | Volume de deplacement |
| Bwl | Bwl_m | m | Bau a la flottaison |
| Tc | Tc_m | m | Tirant d'eau canoe body |
| Swet | Swet_m2 | m2 | Surface mouillee |
| Tmax | Tmax_m | m | Tirant d'eau total |
| Amax | Amax_m2 | m2 | Section immergee maximale |
| Î”m | displacement_mass_kg | kg | Compare a rho_water * volume |
| Ff | Ff_m | m | Franc-bord avant stocke |
| Fa | Fa_m | m | Franc-bord arriere stocke |
| Boa | Boa_m | m | Bau hors tout |
| Loa | Loa_m | m | Longueur hors tout |
| ks | hull_roughness_ks_um | um | Stocke, correction rugosite non implementee |
| P | P_m | m | Hauteur de grand-voile/aile |
| E | E_m | m | Bordure |
| Roach | roach | - | Coefficient sans dimension |
| BAD | BAD_m | m | Hauteur de bome/emplanture |
| Lkeel | Lkeel_m | m | Corde/longueur caracteristique |
| Dkeel | Dkeel_m | m | Profondeur/span |
| t/c | thickness_ratio | - | Epaisseur relative |
| Cu | rudder_Cu_m | m | Corde superieure safran |
| Cl | rudder_Cl_m | m | Corde inferieure safran |
| b | rudder_b_m | m | Profondeur safran |
| TWA sweep | TWA_RANGE_MIN_DEG / TWA_RANGE_MAX_DEG / TWA_RANGE_STEPS | deg / steps | Domaine global de CAP/TWA, bornes incluses |
| Heel sweep | HEEL_RANGE_MIN_DEG / HEEL_RANGE_MAX_DEG / HEEL_RANGE_STEPS | deg / steps | Domaine global de gite et courbe GZ, bornes incluses |
| TWS range | tws_min / tws_max | knots ou m/s | Converti en m/s en interne |
| Hs | Hs_m | m | Stocke, Raw=0 si vagues non implementees |
| Ts | Ts_s | s | Stocke, Raw=0 si vagues non implementees |

## Modele aero manuel

Dans l'interface automatique, `Aero model` permet aussi de choisir un modele Pedro sans remplacer la geometrie STL ni l'hydrodynamique calculee automatiquement. Le panneau Pedro demande le plan de voilure, le jeu de voiles, le greement, les reglages `flat`/`reef` et les dimensions de franc-bord/largeur utilisees par le modele de trainee. L'equilibre utilise alors le meme calcul Pedro qu'en mode Manual; les champs alpha NACA sont ignores. Les options de flat et reef peuvent etre optimisees separement.

`sail_coefficient_data_source = "naca"` force l'ancien modele `naca_wing`: l'aile/grand-voile est traitee comme un profil NACA et le solveur optimise `alpha_wing_deg` si le mode alpha est automatique.

`sail_coefficient_data_source = "hazen"` selectionne `pedro_hazen`; `"orc"` selectionne `pedro_orc_low` par defaut, sauf si `advanced.aero_model` demande explicitement `pedro_orc_high` ou `pedro_orc_medium`. Les modeles Pedro utilisent les tables CL/CD Hazen ou ORC et les champs `rig` / `sailset` pour calculer `A_main`, `A_jib`, `A_spi`, `A_ref` et `zCoE`.

Les tables Pedro sont interpolees avec une interpolation PCHIP monotone par morceaux. En configuration `main_jib`, le foc recoit aussi une enveloppe AWA progressive: coupe sous 10 deg, pleine entre 25 et 120 deg, puis recoupee avant le plein portant.

Dans les modeles Pedro, `flat` reduit le lift et `reef` reduit a la fois le lift, le drag de profil et le centre d'effort. Dans l'onglet Pedro Aero, les cases `Auto-optimize flat` et `Auto-optimize reef` activent separement le balayage de `flat_min..1` et `reef_min..1` (8 valeurs par parametre); si les deux sont cochees, le solveur teste les 64 combinaisons. Un parametre dont l'optimisation est desactivee garde sa valeur fixe. Les anciens JSON avec `optimize_flat_reef` activent les deux balayages. Le resultat expose les colonnes `best_flat`, `best_reef`, `CL_main`, `CD_main`, `A_ref`, `A_main`, `zCoE_sailplan_m`, `coefficient_source` et `interpolation_method`.

Pour les configurations avec voile d'avant (`main_jib` ou `main_spinnaker`), l'interface synchronise `Sail type` et `Sail set` puis active automatiquement les optimisations de flat et de reef. Sans cette optimisation, le jeu de voiles reste plein, le moment de gite et la trainee induite peuvent dominer au pres, et la polaire peut artificiellement s'effondrer.

## Réglages recommandés pour comparaison Pedro/OceanVPP — grande voile seule

Pour comparer ARGO-VPP à Pedro avec seulement la grande voile :

- Sail data = `orc`
- Aero model = `pedro_orc_low`
- Sail set = `main_only`
- Reference area = `active_area`
- Hydro = `oceanvpp_delft`
- Solver = `least_squares_multistart`
- `flat = 1.0`
- `reef = 1.0`
- `optimize_flat = False` et `optimize_reef = False`
- `pedro_hardcoded_hydroheel = True`

`active_area` impose `A_ref = A_main`. À l'inverse, `pedro_nominal` utilise `A_main + A_fore` et ne représente donc pas une normalisation strictement grande voile seule. `naca_wing` reste destiné à l'aile rigide Microtransat, pas à cette comparaison Pedro. Enfin, `orc_simplified` ne constitue pas un VPP ORC complet.

`pedro_hardcoded` reproduit le comportement OceanVPP/Pedro ; `argo_geometric_CLR` rend le bras de levier géométrique plus explicite pour ARGO-VPP.

## Limites du modele

- ARGO-VPP reste un VPP 2-DOF.
- Pas de resolution explicite de derive/leeway.
- Pas d'equilibre de lacet.
- Pas de gouvernail actif.
- Pas de modele de vagues complet: `R_added_waves_N` reste nul si Raw n'est pas implemente.
- Le mode ORC est simplifie.
- Les resultats servent a comparer des tendances, pas a produire une certification ORC.

## 18. Tests numeriques et validation

Les tests sont executables de haut en bas. Ceux qui dependent d'un STL absent sont signales comme `SKIP` plutot que masques.

In [ ]:
def assert_close(a: float, b: float, tol: float, message: str) -> None:
    if abs(float(a) - float(b)) > tol:
        raise AssertionError(f"{message}: {a} != {b} (tol={tol})")


# Test suite replaced by FINAL_NUMERICAL_TESTS below.

# BEGIN FINAL_NUMERICAL_TESTS
def _synthetic_foil(name: str = "derive") -> FoilGeometry:
    contour = np.array([[0.0, 0.0], [0.2, 0.0], [0.18, 0.9], [0.04, 0.95], [0.0, 0.0]], dtype=float)
    area, centroid2, mean_chord, bands = planform_properties(contour)
    mesh = trimesh.creation.box(extents=[0.2, 0.02, 0.9])
    mesh.apply_translation([0.0, 0.0, -0.45])
    return FoilGeometry(name, contour, "NACA0010", area, float(mesh.area), float(mesh.area), 0.95, mean_chord, 0.001, np.array([0.0, 0.0, -0.45]), np.array([0.0, 0.0, -0.2]), 5.0, 0.10, np.zeros(3), mesh, bands)


def _synthetic_state() -> BoatGeometryState:
    tables = load_delft_tables()
    hydro = HullHydrostatics(
        LWL=2.0, BWL=0.5, TC=0.2, volume_C=0.0784, S_C=0.9, A_WP=0.8,
        LCB=0.90, LCF=0.82, AMS_CB=0.07, C_M=0.70, C_P=0.56, x_FP=2.0,
        LCB_fpp=1.10, LCF_fpp=1.18, LCB_pct_midship=-5.0, LCF_pct_midship=-9.0,
        waterline_z=0.0, B_CB=0.5, delft_valid=True, delft_warnings=[],
    )
    wing = _synthetic_foil("aile")
    wing = FoilGeometry("aile", wing.contour, "NACA0012", 1.2, 2.4, 2.4, 2.0, 0.6, 0.01, np.array([0.9, 0.0, 1.0]), np.array([0.9, 0.0, 1.0]), 3.3, 0.12, np.array([0.9, 0.0, 0.0]), wing.mesh, wing.chord_sections)
    cb = _synthetic_foil("derive")
    rud = _synthetic_foil("safran")
    comp = pd.DataFrame([
        {"component": "coque", "mass_kg": 70.0, "x_G_m": 1.0, "y_G_m": 0.0, "z_G_m": -0.1, "area_m2": 1.0},
        {"component": "aile", "mass_kg": 5.0, "x_G_m": 0.9, "y_G_m": 0.0, "z_G_m": 1.0, "area_m2": 2.4},
        {"component": "ballast ponctuel de quille", "mass_kg": 20.0, "x_G_m": 1.0, "y_G_m": 0.0, "z_G_m": -0.6, "area_m2": np.nan},
    ])
    total_mass = float(comp["mass_kg"].sum())
    cg = np.average(comp[["x_G_m", "y_G_m", "z_G_m"]].to_numpy(float), axis=0, weights=comp["mass_kg"].to_numpy(float))
    masses = MassProperties(comp, total_mass, cg, 20.0, np.array([1.0, 0.0, -0.6]), total_mass, 0.0, 0.0)
    gz = pd.DataFrame({
        "heel_deg": [0.0, 2.0, 5.0, 30.0],
        "phi_deg": [0.0, 2.0, 5.0, 30.0],
        "phi_rad": np.deg2rad([0.0, 2.0, 5.0, 30.0]),
        "waterline_z_m": [0.0, 0.01, 0.02, 0.06],
        "volume_m3": [0.08, 0.08, 0.08, 0.08],
        "B_x_m": [1.0, 1.0, 1.0, 1.0],
        "B_y_m": [0.0, 0.01, 0.02, 0.05],
        "B_z_m": [-0.05, -0.05, -0.05, -0.05],
        "GZ_signed_m": [0.0, 0.02, 0.05, -0.02],
        "M_hydrostatic_raw_signed_Nm": [0.0, -20.0, -50.0, 20.0],
        "M_hydrostatic_restoring_Nm": [0.0, 20.0, 50.0, -20.0],
        "M_hydrostatic_Nm": [0.0, 20.0, 50.0, -20.0],
    })
    return BoatGeometryState(GeometryInputs(), SolverSettings(max_nfev=30), trimesh.creation.box(), trimesh.creation.box(), hydro, wing, cb, rud, masses, 0.08, 1.0, -1.0, 1.0, 0.4, gz, tables, False, False, 0.0, True, {})


def run_numerical_tests() -> pd.DataFrame:
    rows: list[dict[str, str]] = []

    def record(name: str, status: str, detail: str = "") -> None:
        rows.append({"test": name, "status": status, "detail": detail})

    tables = load_delft_tables()
    state = APP_STATE.get("geometry") or _synthetic_state()
    synth = _synthetic_state()

    tests = []

    def test_low_fn_components():
        V = 0.05 * np.sqrt(C.G * synth.hydro.LWL)
        raw, r, ok, ext, table = bare_hull_residuary(V, synth.hydro, tables)
        if not ok or not ext or r < 0:
            raise AssertionError("low Fn hull extension not active")
    tests.append(("Test 1 - faible nombre de Froude", test_low_fn_components))

    def test_low_fn_extension():
        lo, _ = fn_domain(tables, "bare_hull")
        V0 = 0.0
        Vlo = lo * np.sqrt(C.G * synth.hydro.LWL)
        Vmid = 0.5 * Vlo
        _, r0, _, _, _ = bare_hull_residuary(V0, synth.hydro, tables)
        _, rlo, _, _, _ = bare_hull_residuary(Vlo, synth.hydro, tables)
        _, rmid, _, _, _ = bare_hull_residuary(Vmid, synth.hydro, tables)
        assert_close(r0, 0.0, 1e-12, "R(Fn=0)")
        assert_close(rmid, 0.5 * rlo, max(1e-9, abs(rlo) * 1e-9), "linear low Fn")
    tests.append(("Test 2 - extension controlee faible Fn", test_low_fn_extension))

    def test_no_high_extrapolation():
        V = 0.8 * np.sqrt(C.G * synth.hydro.LWL)
        _, _, ok, _, reason = bare_hull_residuary(V, synth.hydro, tables)
        if ok or "delft_fn_above_max" not in reason:
            raise AssertionError("Fn above max not rejected")
    tests.append(("Test 3 - absence extrapolation superieure", test_no_high_extrapolation))

    def test_zero_heel():
        S, dRf, ok, _, _ = wetted_surface_heel_delta(1.0, 0.0, synth.hydro, tables)
        dRr, ok2, _, _ = heeled_hull_residuary_delta(1.0, 0.0, synth.hydro, tables)
        if not ok or not ok2:
            raise AssertionError("zero heel invalid")
        assert_close(S, synth.hydro.S_C, 1e-12, "S(phi=0)")
        assert_close(dRf, 0.0, 1e-12, "dRf(phi=0)")
        assert_close(dRr, 0.0, 1e-12, "dRr(phi=0)")
    tests.append(("Test 4 - gite nulle", test_zero_heel))

    def test_low_heel():
        _, dR2, ok2, ext2, _ = wetted_surface_heel_delta(1.0, deg_to_rad(2.0), synth.hydro, tables)
        _, dR5, ok5, _, _ = wetted_surface_heel_delta(1.0, deg_to_rad(5.0), synth.hydro, tables)
        if not ok2 or not ok5 or not ext2:
            raise AssertionError("low heel extension not active")
        if not (min(0.0, dR5) - 1e-9 <= dR2 <= max(0.0, dR5) + 1e-9):
            raise AssertionError("phi=2 deg not between 0 and 5 deg")
    tests.append(("Test 5 - faible gite", test_low_heel))

    def test_solver_heel_bound():
        lo, _ = solver_heel_bounds(synth)
        assert_close(rad_to_deg(lo), 0.0, 1e-12, "heel lower bound")
    tests.append(("Test 6 - borne du solveur", test_solver_heel_bound))

    def test_waterline_heel():
        eq = equilibrium_state(0.4, deg_to_rad(2.0), 8.0, 45.0, synth)
        gz = interpolate_gz_state(synth.gz_curve, 2.0)
        assert_close(eq.z_WL_m, gz["waterline_z_m"], 1e-12, "z_WL(phi)")
    tests.append(("Test 7 - niveau de flottaison avec gite", test_waterline_heel))

    def test_heeling_moment_identity():
        eq = equilibrium_state(0.4, deg_to_rad(2.0), 8.0, 45.0, synth)
        assert_close(eq.M_heel_total_Nm, eq.aero.F_side_N * (eq.z_CE_m - eq.z_CLR_m), 1e-9, "heeling moment identity")
    tests.append(("Test 8 - coherence moment inclinant", test_heeling_moment_identity))

    def test_lcb_conventions():
        h = synth.hydro
        assert_close(h.LCB_fpp / h.LWL, 0.55, 1e-12, "LCB fraction FPP")
        assert_close(h.LCB_pct_midship, -5.0, 1e-12, "LCB pct midship")
        coeff = {"u0": 1, "u1": 0, "u2": 0, "u3": 0, "u4_inferred": 1, "u5": 1}
        frac = h.LCB_fpp / h.LWL
        manual = coeff["u0"] + coeff["u4_inferred"] * frac + coeff["u5"] * frac**2
        wrong = coeff["u0"] + coeff["u4_inferred"] * h.LCB_pct_midship + coeff["u5"] * h.LCB_pct_midship**2
        if abs(manual - wrong) < 1.0:
            raise AssertionError("LCB percent/fraction confusion not detectable")
    tests.append(("Test 9 - convention LCB", test_lcb_conventions))

    def test_mesh_overlap():
        a = trimesh.creation.box(extents=[1, 1, 1])
        b = trimesh.creation.box(extents=[1, 1, 1])
        b.apply_translation([0.25, 0, 0])
        detected, vol = hydrostatic_overlap_diagnostics([a, b])
        if not detected or vol <= 0:
            raise AssertionError("overlap not detected")
    tests.append(("Test 10 - chevauchement de maillage", test_mesh_overlap))

    def test_wing_excluded_buoyancy():
        base = synth.gz_curve.copy()
        modified = synth.gz_curve.copy()
        pd.testing.assert_series_equal(base["waterline_z_m"], modified["waterline_z_m"], check_names=False)
        pd.testing.assert_series_equal(base["M_hydrostatic_restoring_Nm"], modified["M_hydrostatic_restoring_Nm"], check_names=False)
    tests.append(("Test 11 - aile exclue flottabilite", test_wing_excluded_buoyancy))

    def test_wing_mass_included():
        comp = synth.masses.component_table.copy()
        old_cg = synth.masses.center_of_gravity.copy()
        comp.loc[comp["component"] == "aile", "mass_kg"] *= 5.0
        new_cg = np.average(comp[["x_G_m", "y_G_m", "z_G_m"]].to_numpy(float), axis=0, weights=comp["mass_kg"].to_numpy(float))
        if np.linalg.norm(new_cg - old_cg) <= 1e-9:
            raise AssertionError("wing mass does not affect CG")
    tests.append(("Test 12 - aile incluse bilan masse", test_wing_mass_included))

    def test_negative_stability():
        gz = interpolate_gz_state(synth.gz_curve, 30.0)
        if gz["M_hydrostatic_restoring_Nm"] >= 0:
            raise AssertionError("negative branch hidden")
    tests.append(("Test 13 - stabilite negative", test_negative_stability))

    def test_naca_common_domain():
        df = pd.DataFrame({
            "Re": [1.0, 1.0, 2.0, 2.0],
            "alpha_deg": [-10.0, 15.0, -5.0, 12.0],
            "CL": [0.0, 1.0, 0.0, 1.0],
            "CD": [0.01, 0.02, 0.01, 0.02],
            "CM": [0.0, 0.0, 0.0, 0.0],
        })
        ok = interpolate_naca_polar(df, 0.0, 1.5, "synthetic")
        assert_close(ok["alpha_min_valid_deg"], -5.0, 1e-12, "common alpha min")
        assert_close(ok["alpha_max_valid_deg"], 12.0, 1e-12, "common alpha max")
        try:
            interpolate_naca_polar(df, -8.0, 1.5, "synthetic")
            raise AssertionError("alpha outside common domain accepted")
        except PolarDomainError:
            pass
    tests.append(("Test 14 - domaine NACA commun", test_naca_common_domain))

    def test_naca_cache():
        before = NACA_POLAR_CACHE_DIAGNOSTICS["physical_file_reads"]
        cached_naca_polar("NACA0012", "Profil NACA")
        after_first = NACA_POLAR_CACHE_DIAGNOSTICS["physical_file_reads"]
        cached_naca_polar("NACA0012", "Profil NACA")
        after_second = NACA_POLAR_CACHE_DIAGNOSTICS["physical_file_reads"]
        if after_second != after_first:
            raise AssertionError("second cached call reread file")
        if after_first < before:
            raise AssertionError("cache counter invalid")
    tests.append(("Test 15 - cache NACA", test_naca_cache))

    def test_complete_low_fn_case():
        eq = equilibrium_state(0.05 * np.sqrt(C.G * synth.hydro.LWL), deg_to_rad(2.0), 8.0, 60.0, synth)
        if eq.Fn >= 0.25 or not eq.resistance.used_low_fn_physical_extension:
            raise AssertionError("complete low-Fn evaluation did not use controlled extension")
    tests.append(("Test 16 - polaire vent faible Fn<0.25", test_complete_low_fn_case))

    def test_complete_low_heel_case():
        eq = equilibrium_state(0.25 * np.sqrt(C.G * synth.hydro.LWL), deg_to_rad(2.0), 8.0, 60.0, synth)
        if not (0.0 < rad_to_deg(deg_to_rad(2.0)) < 5.0) or not eq.resistance.heel_friction_valid:
            raise AssertionError("complete low-heel evaluation invalid")
    tests.append(("Test 17 - polaire faible gite", test_complete_low_heel_case))

    legacy = [
        ("Vent apparent sans gite", lambda: assert_close(apparent_wind(5.0, 0.0, 73.0, 0.0)[0], 5.0, 1e-12, "AWS")),
        ("Effet de la gite", lambda: assert_close(apparent_wind_components(5.0, 1.0, 70.0, deg_to_rad(25.0))["V_Ay"], apparent_wind_components(5.0, 1.0, 70.0, 0.0)["V_Ay"] * np.cos(deg_to_rad(25.0)), 1e-12, "V_Ay")),
    ]

    for name, func in tests + legacy:
        try:
            func()
            record(name, "OK")
        except Exception as exc:
            record(name, "FAIL", str(exc))

    results = APP_STATE.get("results")
    if isinstance(results, pd.DataFrame) and not results.empty:
        try:
            ok = results[results["solver_success"] == True]
            if not ok.empty:
                if (ok["force_residual_N"].abs() > APP_STATE["geometry"].settings.force_tol_N).any() or (ok["moment_residual_Nm"].abs() > APP_STATE["geometry"].settings.moment_tol_Nm).any():
                    raise AssertionError("residus trop eleves")
                if not ok["naca_valid"].all() or not ok["delft_valid"].all() or not ok["table_domain_valid"].all() or not ok["hydrostatic_geometry_valid"].all():
                    raise AssertionError("solution convergee invalide")
            record("Controle points solver_success", "OK", f"{len(ok)} point(s) valide(s)")
        except Exception as exc:
            record("Controle points solver_success", "FAIL", str(exc))
    else:
        record("Controle points solver_success", "SKIP", "Aucun resultat VPP disponible.")

    df = pd.DataFrame(rows)
    display(df)
    return df


test_report = run_numerical_tests()
# END FINAL_NUMERICAL_TESTS


,test,status,detail
0,Test 1 - faible nombre de Froude,OK,
1,Test 2 - extension controlee faible Fn,OK,
2,Test 3 - absence extrapolation superieure,OK,
3,Test 4 - gite nulle,OK,
4,Test 5 - faible gite,OK,
5,Test 6 - borne du solveur,OK,
6,Test 7 - niveau de flottaison avec gite,OK,
7,Test 8 - coherence moment inclinant,OK,
8,Test 9 - convention LCB,OK,
9,Test 10 - chevauchement de maillage,OK,


In [ ]:
# BEGIN MANUAL_MODE_TESTS
def run_manual_mode_tests() -> pd.DataFrame:
    rows: list[dict[str, str]] = []

    def record(name: str, status: str, detail: str = "") -> None:
        rows.append({"test": name, "status": status, "detail": detail})

    def check(name: str, func: Callable[[], None]) -> None:
        try:
            func()
            record(name, "OK")
        except Exception as exc:
            record(name, "FAIL", str(exc))

    def test_manual_without_stl():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        eq = equilibrium_state(1.0, deg_to_rad(2.0), 8.0 * C.KNOT_TO_MPS, 90.0, state, alpha_override_deg=6.0)
        values = [eq.aero.AWS_mps, eq.force_residual_N, eq.moment_residual_Nm]
        if not all(np.isfinite(value) for value in values):
            raise AssertionError("manual equilibrium smoke test returned non-finite values")

    def test_tws_conversion():
        assert_close(tws_to_mps(8.0, "knots"), 8.0 * C.KNOT_TO_MPS, 1e-12, "8 knots to m/s")

    def test_mass_volume_consistency():
        mi = DEFAULT_MANUAL_PYTHON_VPP_DARING
        mass = C.RHO_WATER * mi.yacht.displaced_volume_m3
        assert_close(mass, 1998.75, 1e-9, "hydrostatic mass")
        spec = process_specifications(mi)
        major = [w for w in spec["warnings"] if "Mass-volume mismatch" in w]
        if major:
            raise AssertionError("Daring should not trigger a major mass-volume warning")

    def test_sail_area():
        spec = process_specifications(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        if is_pedro_aero_model(spec["aero_model"]):
            expected = compute_pedro_sail_geometry(DEFAULT_MANUAL_PYTHON_VPP_DARING)["A_ref"]
        else:
            s = DEFAULT_MANUAL_PYTHON_VPP_DARING.main_sail
            expected = 0.5 * s.P_m * s.E_m * (1.0 + s.roach)
        assert_close(spec["sail_area_m2"], expected, 1e-12, "manual sail area")

    def test_rudder_area():
        r = DEFAULT_MANUAL_PYTHON_VPP_DARING.rudder
        expected = 2.0 * 0.5 * (r.Cu_m + r.Cl_m) * r.b_m
        spec = process_specifications(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        assert_close(spec["rudder_wetted_area_m2"], expected, 1e-12, "manual rudder wetted area")

    def test_speed_bound_restrictive():
        tables = {
            "bare_hull": pd.DataFrame({"Fn": [0.0, 0.70]}),
            "keel": pd.DataFrame({"Fn": [0.0, 0.55]}),
            "heeled_residuary": pd.DataFrame({"Fn": [0.0, 0.62]}),
        }
        state = _synthetic_state()
        state.delft_tables = tables
        _, hi = solver_speed_bounds(state)
        expected = 0.55 * np.sqrt(C.G * state.hydro.LWL)
        assert_close(hi, expected, 1e-12, "restrictive Fn upper bound")

    def test_lcb_pct_midship():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        assert_close(state.hydro.LCB_pct_midship, -4.0, 1e-12, "manual LCB pct")
        Fn = 0.45
        V = Fn * np.sqrt(C.G * state.hydro.LWL)
        dR, ok, _, _ = heeled_hull_residuary_delta(V, deg_to_rad(10.0), state.hydro, state.delft_tables)
        if not ok or not np.isfinite(dR):
            raise AssertionError("LCB pct heeled residuary evaluation failed")

    def test_heeled_hull_table_scaling_and_sign():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        row_045 = bounded_table_row(state.delft_tables, "heeled_residuary", "Fn", 0.45)
        assert_close(row_045["u4_inferred"], -1.0064, 1e-12, "DSYHS u4 at Fn=0.45")
        V = 0.45 * np.sqrt(C.G * state.hydro.LWL)
        dR, ok, _, _ = heeled_hull_residuary_delta(V, deg_to_rad(10.0), state.hydro, state.delft_tables)
        if not ok:
            raise AssertionError("signed heeled hull delta invalid")
        assert_close(dR, 3.3664466130405013, 1e-9, "heeled hull delta scale at Fn=0.45")
        V_high = 0.517217033 * np.sqrt(C.G * state.hydro.LWL)
        dR_high, ok_high, _, _ = heeled_hull_residuary_delta(V_high, deg_to_rad(16.277964), state.hydro, state.delft_tables)
        if not ok_high or not (-1.0 < dR_high < 0.0):
            raise AssertionError(f"expected a small negative signed delta near Fn=0.517, got {dR_high}")

    def test_bare_hull_residuary_scale():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        V = 0.35 * np.sqrt(C.G * state.hydro.LWL)
        raw, value, ok, _, _ = bare_hull_residuary(V, state.hydro, state.delft_tables)
        if not ok:
            raise AssertionError("bare hull residuary evaluation failed")
        assert_close(raw, 177.00855590621794, 1e-9, "bare hull residuary scale at Fn=0.35")
        assert_close(value, raw, 1e-12, "positive bare hull residuary value")

    def test_wind_gradient_changes_result():
        mi0 = clone_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        mi1 = clone_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        mi0.advanced.use_wind_gradient = False
        mi1.advanced.use_wind_gradient = True
        s0 = build_model_from_manual_input(mi0)
        s1 = build_model_from_manual_input(mi1)
        a0 = aerodynamic_state(8.0 * C.KNOT_TO_MPS, 90.0, 1.0, 0.0, s0, alpha_override_deg=6.0)
        a1 = aerodynamic_state(8.0 * C.KNOT_TO_MPS, 90.0, 1.0, 0.0, s1, alpha_override_deg=6.0)
        if abs(a0.AWS_mps - a1.AWS_mps) <= 1e-9:
            raise AssertionError("wind gradient did not change AWS")

    def test_section_area_waterline_clip():
        box = trimesh.creation.box(extents=[2.0, 2.0, 2.0])
        area = section_area_at_x(box, 0.0, 0.0)
        assert_close(area, 2.0, 1e-9, "clipped half-box section area")

    def test_manual_gz_nan_b_columns_do_not_block():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        gz = interpolate_gz_state(state.gz_curve, 2.0)
        if not np.isfinite(gz["M_hydrostatic_restoring_Nm"]):
            raise AssertionError("manual GZ interpolation failed on restoring moment")
        if not np.isnan(gz.get("B_x_m", np.nan)):
            raise AssertionError("manual placeholder B_x_m should remain NaN")

    def test_pedro_hazen_main_coefficients():
        coeff = get_pedro_sail_coefficients("pedro_hazen", 50.0, "main")
        assert_close(coeff["CL"], 1.50, 1e-12, "Hazen main CL at AWA=50")
        assert_close(coeff["CDp"], 0.15, 1e-12, "Hazen main CD at AWA=50")

    def test_pedro_orc_low_main_coefficients():
        coeff = get_pedro_sail_coefficients("pedro_orc_low", 28.0, "main")
        assert_close(coeff["CL"], 1.34698, 1e-12, "ORC low main CL at AWA=28")
        assert_close(coeff["CDp"], 0.03259, 1e-12, "ORC low main CD at AWA=28")

    def test_pedro_orc_high_main_coefficients():
        coeff = get_pedro_sail_coefficients("pedro_orc_high", 12.0, "main")
        assert_close(coeff["CL"], 1.25, 1e-12, "ORC high main CL at AWA=12")
        assert_close(coeff["CDp"], 0.01466, 1e-12, "ORC high main CD at AWA=12")

    def test_pedro_orc_medium_formula():
        fractionality = 0.9
        fcoef, warning = pedro_fractionality_coefficient(fractionality)
        if warning:
            raise AssertionError(warning)
        low = get_pedro_sail_coefficients("pedro_orc_low", 28.0, "main", fractionality=fractionality)
        high = get_pedro_sail_coefficients("pedro_orc_high", 28.0, "main", fractionality=fractionality)
        medium = get_pedro_sail_coefficients("pedro_orc_medium", 28.0, "main", fractionality=fractionality)
        expected_cl = low["CL"] * (1.0 - fcoef / 2.0) + high["CL"] * (fcoef / 2.0)
        expected_cd = low["CDp"] * (1.0 - fcoef / 2.0) + high["CDp"] * (fcoef / 2.0)
        assert_close(medium["CL"], expected_cl, 1e-12, "ORC medium CL formula")
        assert_close(medium["CDp"], expected_cd, 1e-12, "ORC medium CD formula")

    def test_pedro_interpolation_is_shape_preserving():
        coeff = get_pedro_sail_coefficients("pedro_orc_low", 8.0, "jib")
        if not (0.0 <= coeff["CL"] <= 1.0):
            raise AssertionError(f"ORC jib interpolation overshot between AWA 7 and 9 deg: {coeff['CL']}")
        if "CubicSpline" in coeff["interpolation_method"]:
            raise AssertionError("Pedro interpolation must avoid CubicSpline overshoot on sparse sail tables")

    def test_pedro_jib_usage_envelope():
        assert_close(pedro_headsail_usage_factor(5.0), 0.0, 1e-12, "jib inactive at tiny AWA")
        assert_close(pedro_headsail_usage_factor(30.0), 1.0, 1e-12, "jib fully active at working AWA")
        assert_close(pedro_headsail_usage_factor(155.0), 0.0, 1e-12, "jib inactive deep downwind")

    def test_pedro_sail_geometry():
        geom = compute_pedro_sail_geometry(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        assert_close(geom["A_main"], 17.82, 1e-12, "Pedro A_main")
        assert_close(geom["zCoE_main"], 5.012, 1e-12, "Pedro zCoE_main")

    def test_pedro_force_direction():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        TWS = 8.0 * C.KNOT_TO_MPS
        aero = aerodynamic_state(TWS, 90.0, TWS, 0.0, state)
        assert_close(aero.AWA_deg, 45.0, 1e-12, "Pedro force direction AWA")
        awa_rad = deg_to_rad(aero.AWA_deg)
        q = 0.5 * C.RHO_AIR * aero.AWS_mps**2
        expected_drive = q * aero.A_ref * (aero.CL * np.sin(awa_rad) - aero.CD * np.cos(awa_rad))
        expected_side = q * aero.A_ref * (aero.CL * np.cos(awa_rad) + aero.CD * np.sin(awa_rad))
        assert_close(aero.F_drive_N, expected_drive, 1e-9, "Pedro drive direction")
        assert_close(aero.F_side_signed_N, expected_side, 1e-9, "Pedro side-force direction")
        if aero.F_drive_N <= 0.0 or aero.F_side_N <= 0.0:
            raise AssertionError("Pedro force direction should produce positive drive and side force at this point")

    def test_pedro_does_not_call_naca_cache():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        before = dict(NACA_POLAR_CACHE_DIAGNOSTICS)
        original = globals()["cached_naca_polar"]
        def fail_cached_naca_polar(*args: Any, **kwargs: Any):
            raise AssertionError("Pedro aerodynamic model unexpectedly called cached_naca_polar")
        globals()["cached_naca_polar"] = fail_cached_naca_polar
        try:
            aerodynamic_state(8.0 * C.KNOT_TO_MPS, 90.0, 1.0, 0.0, state)
        finally:
            globals()["cached_naca_polar"] = original
        if NACA_POLAR_CACHE_DIAGNOSTICS != before:
            raise AssertionError("Pedro aerodynamic model changed the NACA cache diagnostics")

    def test_pedro_result_columns():
        required = {
            "aero_model", "sailset", "reference_area_mode", "best_flat", "best_reef", "flat", "reef",
            "flat_reef_candidate_count", "flat_reef_valid_candidate_count",
            "CL_main", "CD_main", "CL_jib", "CD_jib", "CL_spin", "CD_spin",
            "CL_lift_weighted", "CDp_weighted", "CDi", "CD0", "CD_total",
            "A_ref", "A_main", "A_ref_m2", "A_main_m2", "A_jib", "A_fore", "A_spi",
            "heeling_moment_model", "hydro_model", "solver_method", "appendage_lift_drag_model", "warning_message",
            "zCoE_sailplan_m", "coefficient_source", "interpolation_method",
        }
        missing = sorted(required.difference(RESULT_COLUMNS))
        if missing:
            raise AssertionError(f"Missing Pedro result columns: {missing}")

    def test_grid_progress_callback():
        state = build_model_from_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        events: list[tuple[int, int, float, float, bool]] = []
        original_solver = globals()["solve_condition_with_alpha_mode"]
        original_row_builder = globals()["row_from_solution"]

        def fake_solver(*args: Any, **kwargs: Any) -> dict[str, Any]:
            return {"success": True, "boat_speed_mps": 1.0, "heel_rad": 0.0}

        globals()["solve_condition_with_alpha_mode"] = fake_solver
        globals()["row_from_solution"] = lambda *args, **kwargs: {}
        try:
            result = solve_vpp_grid(
                [8.0, 12.0], state, twa_values_deg=[35.0, 40.0],
                progress_callback=lambda *event: events.append(event),
            )
        finally:
            globals()["solve_condition_with_alpha_mode"] = original_solver
            globals()["row_from_solution"] = original_row_builder
        expected = [
            (1, 4, 8.0, 35.0, True), (2, 4, 8.0, 40.0, True),
            (3, 4, 12.0, 35.0, True), (4, 4, 12.0, 40.0, True),
        ]
        if events != expected or len(result) != 4:
            raise AssertionError(f"unexpected grid progress events: {events}")

    def test_manual_progress_widget():
        ui = build_manual_input_widgets()
        progress_widgets = [child for child in ui.children[-1].children if isinstance(child, widgets.IntProgress)]
        if len(progress_widgets) != 1:
            raise AssertionError("manual UI must contain exactly one progress bar")
        if progress_widgets[0].layout.display != "none":
            raise AssertionError("progress bar must be hidden before the first calculation")

    def test_aero_selection_normalization():
        for model, expected_source in [("pedro_orc_low", "orc"), ("pedro_hazen", "hazen"), ("naca_wing", "naca")]:
            normalized = normalize_aero_selection({"aero_model": model, "sail_coefficient_data_source": "naca" if expected_source != "naca" else "orc"})
            if normalized["aero_model"] != model or normalized["sail_coefficient_data_source"] != expected_source:
                raise AssertionError(f"incoherent normalized selection: {normalized}")
            if not normalized.get("warning_message"):
                raise AssertionError("automatic correction must add a warning")

    def test_custom_refused():
        try:
            normalize_aero_selection({"aero_model": "pedro_orc_low", "sail_coefficient_data_source": "custom"})
            raise AssertionError("custom sail data was accepted")
        except ValueError as exc:
            if "not implemented" not in str(exc):
                raise

    def test_main_only_active_area():
        mi = clone_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        mi.sailset.sailset = "main_only"
        mi.sailset.reference_area_mode = "active_area"
        geom = compute_pedro_sail_geometry(mi)
        assert_close(geom["A_main"], 17.82, 1e-12, "mainsail area")
        assert_close(geom["A_ref"], 17.82, 1e-12, "active reference area")

    def test_pedro_nominal_warning():
        mi = clone_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING)
        mi.sailset.sailset = "main_only"
        mi.sailset.reference_area_mode = "pedro_nominal"
        warnings_text = "; ".join(compute_pedro_sail_geometry(mi)["warnings"])
        if "main_only with pedro_nominal" not in warnings_text:
            raise AssertionError("missing pedro_nominal warning")

    def test_negative_total_resistance_preserved():
        state = _synthetic_state()
        names = ["hull_viscous_resistance", "orc_appendage_viscous", "bare_hull_residuary", "centerboard_residuary", "wetted_surface_heel_delta", "heeled_hull_residuary_delta", "heeled_keel_residuary_delta", "induced_hydrodynamic_drag"]
        original = {name: globals()[name] for name in names}
        globals()["hull_viscous_resistance"] = lambda *a, **k: (0.0, [])
        globals()["orc_appendage_viscous"] = lambda *a, **k: 0.0
        globals()["bare_hull_residuary"] = lambda *a, **k: (0.0, 0.0, True, False, "")
        globals()["centerboard_residuary"] = lambda *a, **k: (0.0, True, False, "")
        globals()["wetted_surface_heel_delta"] = lambda *a, **k: (state.hydro.S_C, -5.0, True, False, "")
        globals()["heeled_hull_residuary_delta"] = lambda *a, **k: (0.0, True, False, "")
        globals()["heeled_keel_residuary_delta"] = lambda *a, **k: (0.0, True, False, "")
        globals()["induced_hydrodynamic_drag"] = lambda *a, **k: (0.0, 1.0, 1.0, True, "")
        try:
            rb = resistance_breakdown(1.0, 0.0, 0.0, state)
        finally:
            globals().update(original)
        assert_close(rb.R_total_N, -5.0, 1e-12, "signed total resistance")
        if "Computed total resistance is non-positive" not in rb.warning_message or rb.delft_valid:
            raise AssertionError("non-positive total must be warned and invalid")

    def test_invalid_plot_speed_is_nan():
        prepared = prepare_plot_dataframe(pd.DataFrame({"boat_speed_kn": [4.2, 0.0], "validity_flag": ["valid", "invalid"]}))
        if not np.isfinite(prepared.loc[0, "boat_speed_plot"]) or not np.isnan(prepared.loc[1, "boat_speed_plot"]):
            raise AssertionError("invalid plot speed was not converted to NaN")

    def test_solver_menu_and_behavior():
        ui = build_manual_input_widgets()
        solver_widgets = [child for child in ui.children[0].children if isinstance(child, widgets.Dropdown) and child.description == "Solver"]
        if len(solver_widgets) != 1 or set(solver_widgets[0].options) != {"least_squares_multistart", "least_squares_single"}:
            raise AssertionError(f"unexpected solver menu: {solver_widgets[0].options if solver_widgets else None}")
        state = _synthetic_state()
        calls: list[int] = []
        original_least_squares = opt.least_squares
        opt.least_squares = lambda *a, **k: (calls.append(1), (_ for _ in ()).throw(RuntimeError("test stop")))[1]
        try:
            solve_one_condition(4.0, 60.0, state, solver_method="least_squares_single")
            single_calls = len(calls)
            calls.clear()
            solve_one_condition(4.0, 60.0, state, solver_method="least_squares_multistart")
            multi_calls = len(calls)
        finally:
            opt.least_squares = original_least_squares
        if single_calls != 1 or multi_calls <= single_calls:
            raise AssertionError(f"solver choices do not change seed count: single={single_calls}, multi={multi_calls}")

    def test_pedro_flat_reef_optimization_toggles():
        original_solve_one_condition = globals()["solve_one_condition"]
        for optimize_flat, optimize_reef, legacy_optimize, expected_count in [(False, False, False, 1), (True, False, False, 8), (False, True, False, 8), (True, True, False, 64), (False, False, True, 64)]:
            sailset = SailSetInput(optimize_flat=optimize_flat, optimize_reef=optimize_reef, optimize_flat_reef=legacy_optimize)
            state = type("TestState", (), {})()
            state.settings = type("TestSettings", (), {"aero_model": "pedro_orc_low"})()
            state.manual_input = ManualVPPInput(sailset=sailset)
            candidates: list[tuple[float, float]] = []

            def fake_solve_one_condition(*args: Any, flat_override: float | None = None, reef_override: float | None = None, **kwargs: Any) -> dict[str, Any]:
                candidates.append((float(flat_override), float(reef_override)))
                return {"success": True, "boat_speed_mps": 4.0, "heel_rad": 0.1}

            globals()["solve_one_condition"] = fake_solve_one_condition
            try:
                solution = solve_condition_with_alpha_mode(5.0, 60.0, state)
            finally:
                globals()["solve_one_condition"] = original_solve_one_condition
            if len(candidates) != expected_count or solution["flat_reef_candidate_count"] != expected_count:
                raise AssertionError(f"unexpected Pedro candidate count for flat={optimize_flat}, reef={optimize_reef}: {len(candidates)}")
            flat_values = {flat_value for flat_value, _reef_value in candidates}
            reef_values = {reef_value for _flat_value, reef_value in candidates}
            if ((optimize_flat or legacy_optimize) and len(flat_values) != 8) or (not (optimize_flat or legacy_optimize) and flat_values != {1.0}):
                raise AssertionError(f"flat optimization toggle ignored: {flat_values}")
            if ((optimize_reef or legacy_optimize) and len(reef_values) != 8) or (not (optimize_reef or legacy_optimize) and reef_values != {1.0}):
                raise AssertionError(f"reef optimization toggle ignored: {reef_values}")

    for name, func in [
        ("Manual input without STL", test_manual_without_stl),
        ("TWS unit conversion", test_tws_conversion),
        ("Mass-volume consistency", test_mass_volume_consistency),
        ("Manual sail area", test_sail_area),
        ("Manual rudder area", test_rudder_area),
        ("Restrictive speed bound", test_speed_bound_restrictive),
        ("LCB pct midship", test_lcb_pct_midship),
        ("Heeled hull table scaling and sign", test_heeled_hull_table_scaling_and_sign),
        ("Bare hull residuary scale", test_bare_hull_residuary_scale),
        ("Wind gradient", test_wind_gradient_changes_result),
        ("Section area waterline clipping", test_section_area_waterline_clip),
        ("Manual GZ NaN B columns", test_manual_gz_nan_b_columns_do_not_block),
        ("Pedro Hazen main coefficients", test_pedro_hazen_main_coefficients),
        ("Pedro ORC low main coefficients", test_pedro_orc_low_main_coefficients),
        ("Pedro ORC high main coefficients", test_pedro_orc_high_main_coefficients),
        ("Pedro ORC medium formula", test_pedro_orc_medium_formula),
        ("Pedro shape-preserving interpolation", test_pedro_interpolation_is_shape_preserving),
        ("Pedro jib usage envelope", test_pedro_jib_usage_envelope),
        ("Pedro sail geometry", test_pedro_sail_geometry),
        ("Pedro force direction", test_pedro_force_direction),
        ("Pedro does not call NACA cache", test_pedro_does_not_call_naca_cache),
        ("Pedro result columns", test_pedro_result_columns),
        ("Grid progress callback", test_grid_progress_callback),
        ("Manual progress widget", test_manual_progress_widget),
        ("Aero/Sail data normalization", test_aero_selection_normalization),
        ("Custom sail data refused", test_custom_refused),
        ("Main-only active reference area", test_main_only_active_area),
        ("Pedro nominal warning", test_pedro_nominal_warning),
        ("Negative total resistance preserved", test_negative_total_resistance_preserved),
        ("Invalid plot speed is NaN", test_invalid_plot_speed_is_nan),
        ("Solver menu and behavior", test_solver_menu_and_behavior),
        ("Pedro flat/reef optimization toggles", test_pedro_flat_reef_optimization_toggles),
    ]:
        check(name, func)

    df = pd.DataFrame(rows)
    display(df)
    return df


manual_test_report = run_manual_mode_tests()
# END MANUAL_MODE_TESTS

c:\Users\erikg\Documents\HEIA\Master\TM\01. Logiciel\ARGOVPP\fonctions\argo_vpp_nb_solver.py:1780: UserWarning: solver_speed_bounds: missing table(s) bare_hull, keel, heeled_residuary; using fallback Fn<=0.80.
  warnings.warn(f"solver_speed_bounds: missing table(s) {', '.join(missing)}; using fallback Fn<={fallback_hi:.2f}.")


,test,status,detail
0,Manual input without STL,OK,
1,TWS unit conversion,OK,
2,Mass-volume consistency,OK,
3,Manual sail area,OK,
4,Manual rudder area,OK,
5,Restrictive speed bound,OK,
6,LCB pct midship,OK,
7,Heeled hull table scaling and sign,OK,
8,Bare hull residuary scale,OK,
9,Wind gradient,OK,


In [ ]:
# BEGIN PLOT_FAILURE_CHECKUP
def run_plot_failure_checkup(
    manual_input: ManualVPPInput | None = None,
    tws_values: tuple[float, ...] = (8.0, 12.0),
    twa_values: tuple[float, ...] = (35.0, 90.0),
    show_plot: bool = False,
) -> dict[str, Any]:
    """Diagnose why the speed polar is empty without changing solver code."""

    mi = clone_manual_input(DEFAULT_MANUAL_PYTHON_VPP_DARING if manual_input is None else manual_input)
    state = build_model_from_manual_input(mi)
    candidates, candidate_count = alpha_candidates_for_condition(state)
    polar_path, polar_df = cached_naca_polar(state.inputs.wing_naca, state.inputs.polar_folder)
    speed_lo, speed_hi = solver_speed_bounds(state)

    summary_rows = [
        {"check": "geometry_input_mode", "value": state.settings.geometry_input_mode},
        {"check": "sail_source_requested", "value": mi.solver.sail_coefficient_data_source},
        {"check": "polar_file", "value": str(polar_path)},
        {"check": "polar_Re_min", "value": float(polar_df["Re"].min())},
        {"check": "polar_Re_max", "value": float(polar_df["Re"].max())},
        {"check": "alpha_candidate_count", "value": candidate_count},
        {"check": "alpha_candidates_preview", "value": candidates[:5] + (["..."] if len(candidates) > 10 else []) + candidates[-5:]},
        {"check": "solver_speed_hi_mps", "value": speed_hi},
        {"check": "solver_speed_hi_kn", "value": speed_hi / C.KNOT_TO_MPS},
        {"check": "delft_geometry_valid", "value": state.hydro.delft_valid},
        {"check": "delft_warning_count", "value": len(state.hydro.delft_warnings)},
    ]
    summary = pd.DataFrame(summary_rows)

    reynolds_rows = []
    for tws_value in tws_values:
        TWS_mps = tws_to_mps(float(tws_value), mi.environment.tws_unit)
        for TWA_deg in twa_values:
            for label, V_B in [("lo", speed_lo), ("mid", 0.5 * speed_hi), ("hi", speed_hi)]:
                AWS, AWA = apparent_wind(TWS_mps, float(V_B), float(TWA_deg), 0.0)
                Re = safe_div(C.RHO_AIR * AWS * state.wing.mean_chord_m, C.MU_AIR, default=np.nan)
                reynolds_rows.append({
                    "TWS": float(tws_value),
                    "TWA_deg": float(TWA_deg),
                    "speed_probe": label,
                    "V_B_mps": float(V_B),
                    "AWS_mps": AWS,
                    "AWA_deg": AWA,
                    "Re_wing": Re,
                    "Re_inside_polar": bool(float(polar_df["Re"].min()) <= Re <= float(polar_df["Re"].max())),
                })
    reynolds_scan = pd.DataFrame(reynolds_rows)

    df = solve_vpp_grid(
        [float(v) for v in tws_values],
        state,
        twa_values_deg=np.asarray(twa_values, dtype=float),
        tws_unit=mi.environment.tws_unit,
    )
    valid_count = int(df["solver_success"].fillna(False).sum()) if "solver_success" in df else 0
    plot_mask_rows = []
    for tws, group in mirror_polar(df).groupby("TWS_kn"):
        plotted_speed = _valid_plot_group(group, "boat_speed_kn")
        plot_mask_rows.append({
            "TWS_kn": tws,
            "points": len(group),
            "finite_points_sent_to_plot": int(np.isfinite(plotted_speed).sum()),
            "all_points_masked": bool(np.isfinite(plotted_speed).sum() == 0),
        })
    plot_mask = pd.DataFrame(plot_mask_rows)

    diagnostic_cols = [
        "TWS", "TWS_unit", "TWA_deg", "AWA_deg", "AWS_mps", "boat_speed_kn",
        "solver_success", "failure_reason_code", "invalid_reason", "warning_message", "solver_message",
        "alpha_candidate_count", "alpha_valid_candidate_count", "Re_wing", "Re_domain_min", "Re_domain_max",
        "delft_valid", "delft_geometry_valid", "table_domain_valid",
    ]
    diagnostic_cols = [col for col in diagnostic_cols if col in df.columns]
    invalid_counts = df["invalid_reason"].value_counts(dropna=False).rename_axis("invalid_reason").reset_index(name="count") if "invalid_reason" in df else pd.DataFrame()

    print("PLOT FAILURE CHECKUP")
    print(f"Solved rows: {len(df)} ; solver_success rows: {valid_count}")
    if valid_count == 0:
        print("No valid speed reaches plot_speed_polar: _valid_plot_group masks every boat_speed_kn value.")
    display(summary)
    if state.hydro.delft_warnings:
        display(pd.DataFrame({"delft_warning": state.hydro.delft_warnings}))
    display(reynolds_scan)
    display(plot_mask)
    display(invalid_counts)
    display(df[diagnostic_cols])
    if show_plot:
        plot_speed_polar(df)

    report = {
        "state": state,
        "summary": summary,
        "reynolds_scan": reynolds_scan,
        "results": df,
        "plot_mask": plot_mask,
        "invalid_counts": invalid_counts,
    }
    APP_STATE["plot_failure_checkup"] = report
    return report


plot_failure_checkup = None
# Run manually when needed:
# plot_failure_checkup = run_plot_failure_checkup()
# END PLOT_FAILURE_CHECKUP

#Comprehension des problems 

In [ ]:
""" df = APP_STATE.get("results")

if df is None or df.empty:
    print("Aucun résultat disponible : lancez d'abord le calcul VPP.")
else:
    cols = [
        "TWS", "TWA_deg", "solver_success", "boat_speed_kn",
        "failure_reason_code", "invalid_reason", "solver_message",
        "force_residual_N", "moment_residual_Nm",
    ]
    display(df.loc[~df["solver_success"].fillna(False), cols]) """

' df = APP_STATE.get("results")\n\nif df is None or df.empty:\n    print("Aucun résultat disponible : lancez d\'abord le calcul VPP.")\nelse:\n    cols = [\n        "TWS", "TWA_deg", "solver_success", "boat_speed_kn",\n        "failure_reason_code", "invalid_reason", "solver_message",\n        "force_residual_N", "moment_residual_Nm",\n    ]\n    display(df.loc[~df["solver_success"].fillna(False), cols]) '

In [ ]:
""" TWS_DIAG_KN = 10.0
TWA_DIAG_DEG = 90.0

diag_state = APP_STATE.get("geometry")
diag_state_source = "APP_STATE['geometry']"
if not isinstance(diag_state, BoatGeometryState):
    diag_cache = Path("Résultats geometrie") / "ARGO_VPP_geometry_cache.txt"
    if not diag_cache.exists():
        raise RuntimeError("Aucune geometrie active et cache Résultats geometrie/ARGO_VPP_geometry_cache.txt introuvable.")
    diag_state = load_geometry_cache(diag_cache)
    diag_state_source = str(diag_cache)

TWS_DIAG_MPS = tws_to_mps(TWS_DIAG_KN, "knots")
diag_speed_bounds = solver_speed_bounds(diag_state)
diag_heel_bounds = solver_heel_bounds(diag_state)
diag_force_scale = max(0.5 * C.RHO_AIR * TWS_DIAG_MPS**2 * diag_state.wing.planform_area_m2, 1.0)
diag_moment_scale = max(diag_force_scale * max(diag_state.wing.span_m, 1.0), 1.0)
diag_runs = []
diag_evaluations = []
_original_least_squares = opt.least_squares

def _least_squares_with_trace(function, *args, **kwargs):
    run_id = len(diag_runs) + 1
    seed = np.asarray(args[0], dtype=float).copy()
    solver_args = kwargs.get("args", ())
    alpha_value = solver_args[3] if len(solver_args) > 3 else None
    run = {
        "run": run_id,
        "alpha_deg": alpha_value,
        "seed_speed_mps": float(seed[0]),
        "seed_heel_deg": rad_to_deg(float(seed[1])),
        "nfev": 0,
        "success": False,
        "status": np.nan,
        "cost": np.nan,
        "final_speed_mps": np.nan,
        "final_heel_deg": np.nan,
        "message": "",
    }

    def _residual_with_trace(x, *function_args):
        evaluation = len([row for row in diag_evaluations if row["run"] == run_id]) + 1
        try:
            residual = np.asarray(function(x, *function_args), dtype=float)
            diag_evaluations.append({
                "run": run_id,
                "alpha_deg": alpha_value,
                "evaluation": evaluation,
                "speed_mps": float(x[0]),
                "speed_kn": float(x[0]) / C.KNOT_TO_MPS,
                "heel_deg": rad_to_deg(float(x[1])),
                "force_residual_N": float(residual[0] * diag_force_scale),
                "moment_residual_Nm": float(residual[1] * diag_moment_scale),
                "scaled_norm": float(np.linalg.norm(residual)),
                "evaluation_error": "",
            })
            return residual
        except Exception as exc:
            diag_evaluations.append({
                "run": run_id,
                "alpha_deg": alpha_value,
                "evaluation": evaluation,
                "speed_mps": float(x[0]),
                "speed_kn": float(x[0]) / C.KNOT_TO_MPS,
                "heel_deg": rad_to_deg(float(x[1])),
                "force_residual_N": np.nan,
                "moment_residual_Nm": np.nan,
                "scaled_norm": np.nan,
                "evaluation_error": f"{type(exc).__name__}: {exc}",
            })
            raise

    try:
        result = _original_least_squares(_residual_with_trace, *args, **kwargs)
        run.update({
            "nfev": int(result.nfev),
            "success": bool(result.success),
            "status": int(result.status),
            "cost": float(result.cost),
            "final_speed_mps": float(result.x[0]),
            "final_heel_deg": rad_to_deg(float(result.x[1])),
            "message": str(result.message),
        })
        return result
    except Exception as exc:
        run["message"] = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        diag_runs.append(run)

print(f"Geometrie utilisee: {diag_state_source}")
print(f"Cas: TWS={TWS_DIAG_KN:g} kn ({TWS_DIAG_MPS:.6f} m/s), TWA={TWA_DIAG_DEG:g} deg")
print(f"Mode aero: {getattr(diag_state.settings, 'aero_model', 'naca_wing')}; alpha: {diag_state.inputs.wing_alpha_mode}")
print(f"Bornes vitesse: {diag_speed_bounds[0]:.6f} .. {diag_speed_bounds[1]:.6f} m/s")
print(f"Bornes gite: {rad_to_deg(diag_heel_bounds[0]):.3f} .. {rad_to_deg(diag_heel_bounds[1]):.3f} deg")
print(f"Echelles: force={diag_force_scale:.6g} N; moment={diag_moment_scale:.6g} N.m; max_nfev={diag_state.settings.max_nfev}")

opt.least_squares = _least_squares_with_trace
try:
    diag_solution = solve_condition_with_alpha_mode(TWS_DIAG_MPS, TWA_DIAG_DEG, diag_state)
finally:
    opt.least_squares = _original_least_squares

VPP_1KN_90_EVALUATIONS = pd.DataFrame(diag_evaluations)
VPP_1KN_90_RUNS = pd.DataFrame(diag_runs)
print(f"\nSolveur: success={diag_solution.get('success')}; message={diag_solution.get('message')}")
print(f"Code={diag_solution.get('failure_reason_code')}; raison={diag_solution.get('invalid_reason')}")
if diag_solution.get("eq") is not None:
    diag_eq = diag_solution["eq"]
    print(f"Vitesse finale={diag_solution['boat_speed_mps']:.6f} m/s ({diag_solution['boat_speed_mps'] / C.KNOT_TO_MPS:.4f} kn); gite={rad_to_deg(diag_solution['heel_rad']):.4f} deg")
    print(f"Residus finaux: force={diag_eq.force_residual_N:.6g} N; moment={diag_eq.moment_residual_Nm:.6g} N.m")
    print(f"Validite: polaire NACA={diag_eq.naca_valid}; Delft={diag_eq.resistance.delft_valid}; domaine={diag_eq.table_domain_valid}; hydrostatique={diag_eq.hydrostatic_geometry_valid}")
print("\nBilan par depart du solveur")
display(VPP_1KN_90_RUNS)
print("Toutes les evaluations de residus (y compris les perturbations de differences finies du Jacobien)")
display(VPP_1KN_90_EVALUATIONS)
 """

' TWS_DIAG_KN = 10.0\nTWA_DIAG_DEG = 90.0\n\ndiag_state = APP_STATE.get("geometry")\ndiag_state_source = "APP_STATE[\'geometry\']"\nif not isinstance(diag_state, BoatGeometryState):\n    diag_cache = Path("ARGO_VPP_geometry_cache.txt")\n    if not diag_cache.exists():\n        raise RuntimeError("Aucune geometrie active et cache ARGO_VPP_geometry_cache.txt introuvable.")\n    diag_state = load_geometry_cache(diag_cache)\n    diag_state_source = str(diag_cache)\n\nTWS_DIAG_MPS = tws_to_mps(TWS_DIAG_KN, "knots")\ndiag_speed_bounds = solver_speed_bounds(diag_state)\ndiag_heel_bounds = solver_heel_bounds(diag_state)\ndiag_force_scale = max(0.5 * C.RHO_AIR * TWS_DIAG_MPS**2 * diag_state.wing.planform_area_m2, 1.0)\ndiag_moment_scale = max(diag_force_scale * max(diag_state.wing.span_m, 1.0), 1.0)\ndiag_runs = []\ndiag_evaluations = []\n_original_least_squares = opt.least_squares\n\ndef _least_squares_with_trace(function, *args, **kwargs):\n    run_id = len(diag_runs) + 1\n    seed = np.asarr

In [ ]:
""" print("Evaluation directe des points de depart pour reveler les erreurs masquees par le residu penalise")
diag_probe_rows = []
for probe_speed, probe_heel_deg in [
    (max(diag_speed_bounds[0] + 1e-10, 1e-10), 0.0),
    (diag_speed_bounds[1] * 0.15, 0.5),
    (diag_speed_bounds[1] * 0.35, 4.0),
]:
    try:
        probe_eq = equilibrium_state(
            probe_speed,
            deg_to_rad(probe_heel_deg),
            TWS_DIAG_MPS,
            TWA_DIAG_DEG,
            diag_state,
            alpha_override_deg=6.0,
        )
        diag_probe_rows.append({
            "speed_mps": probe_speed,
            "heel_deg": probe_heel_deg,
            "status": "OK",
            "force_residual_N": probe_eq.force_residual_N,
            "moment_residual_Nm": probe_eq.moment_residual_Nm,
            "naca_valid": probe_eq.naca_valid,
            "table_domain_valid": probe_eq.table_domain_valid,
            "delft_valid": probe_eq.resistance.delft_valid,
            "invalid_reason": probe_eq.invalid_reason,
        })
    except Exception as exc:
        diag_probe_rows.append({
            "speed_mps": probe_speed,
            "heel_deg": probe_heel_deg,
            "status": f"{type(exc).__name__}: {exc}",
            "force_residual_N": np.nan,
            "moment_residual_Nm": np.nan,
            "naca_valid": np.nan,
            "table_domain_valid": np.nan,
            "delft_valid": np.nan,
            "invalid_reason": "",
        })
display(pd.DataFrame(diag_probe_rows))
 """

' print("Evaluation directe des points de depart pour reveler les erreurs masquees par le residu penalise")\ndiag_probe_rows = []\nfor probe_speed, probe_heel_deg in [\n    (max(diag_speed_bounds[0] + 1e-10, 1e-10), 0.0),\n    (diag_speed_bounds[1] * 0.15, 0.5),\n    (diag_speed_bounds[1] * 0.35, 4.0),\n]:\n    try:\n        probe_eq = equilibrium_state(\n            probe_speed,\n            deg_to_rad(probe_heel_deg),\n            TWS_DIAG_MPS,\n            TWA_DIAG_DEG,\n            diag_state,\n            alpha_override_deg=6.0,\n        )\n        diag_probe_rows.append({\n            "speed_mps": probe_speed,\n            "heel_deg": probe_heel_deg,\n            "status": "OK",\n            "force_residual_N": probe_eq.force_residual_N,\n            "moment_residual_Nm": probe_eq.moment_residual_Nm,\n            "naca_valid": probe_eq.naca_valid,\n            "table_domain_valid": probe_eq.table_domain_valid,\n            "delft_valid": probe_eq.resistance.delft_valid,\n        

In [ ]:
""" diag_polar_path, diag_polar_df = cached_naca_polar(diag_state.inputs.wing_naca, diag_state.inputs.polar_folder)
print(f"Polaire active: {diag_state.inputs.wing_naca} - {diag_polar_path}")
print(f"Domaine Reynolds de la polaire: {diag_polar_df['Re'].min():.6g} .. {diag_polar_df['Re'].max():.6g}")
for probe_speed, probe_heel_deg in [
    (1e-10, 0.0),
    (diag_speed_bounds[1] * 0.15, 0.5),
    (diag_speed_bounds[1] * 0.35, 4.0),
]:
    probe_aws, probe_awa = apparent_wind(TWS_DIAG_MPS, probe_speed, TWA_DIAG_DEG, deg_to_rad(probe_heel_deg))
    probe_re = C.RHO_AIR * probe_aws * diag_state.wing.mean_chord_m / C.MU_AIR
    print(f"V={probe_speed:.6g} m/s, gite={probe_heel_deg:g} deg: AWS={probe_aws:.6g} m/s, AWA={probe_awa:.4g} deg, Re={probe_re:.6g}")
    try:
        equilibrium_state(probe_speed, deg_to_rad(probe_heel_deg), TWS_DIAG_MPS, TWA_DIAG_DEG, diag_state, alpha_override_deg=6.0)
    except Exception as exc:
        print(f"  Exception exacte: {type(exc).__name__}: {exc}")
 """

' diag_polar_path, diag_polar_df = cached_naca_polar(diag_state.inputs.wing_naca, diag_state.inputs.polar_folder)\nprint(f"Polaire active: {diag_state.inputs.wing_naca} - {diag_polar_path}")\nprint(f"Domaine Reynolds de la polaire: {diag_polar_df[\'Re\'].min():.6g} .. {diag_polar_df[\'Re\'].max():.6g}")\nfor probe_speed, probe_heel_deg in [\n    (1e-10, 0.0),\n    (diag_speed_bounds[1] * 0.15, 0.5),\n    (diag_speed_bounds[1] * 0.35, 4.0),\n]:\n    probe_aws, probe_awa = apparent_wind(TWS_DIAG_MPS, probe_speed, TWA_DIAG_DEG, deg_to_rad(probe_heel_deg))\n    probe_re = C.RHO_AIR * probe_aws * diag_state.wing.mean_chord_m / C.MU_AIR\n    print(f"V={probe_speed:.6g} m/s, gite={probe_heel_deg:g} deg: AWS={probe_aws:.6g} m/s, AWA={probe_awa:.4g} deg, Re={probe_re:.6g}")\n    try:\n        equilibrium_state(probe_speed, deg_to_rad(probe_heel_deg), TWS_DIAG_MPS, TWA_DIAG_DEG, diag_state, alpha_override_deg=6.0)\n    except Exception as exc:\n        print(f"  Exception exacte: {type(exc).__na

In [ ]:
""" diag_max_aws, diag_max_awa = apparent_wind(TWS_DIAG_MPS, diag_speed_bounds[1], TWA_DIAG_DEG, 0.0)
diag_max_re = C.RHO_AIR * diag_max_aws * diag_state.wing.mean_chord_m / C.MU_AIR
print(f"A la vitesse maximale autorisee ({diag_speed_bounds[1]:.6f} m/s), AWS={diag_max_aws:.6f} m/s et Re={diag_max_re:.6g}.")
print(f"La polaire commence a Re={diag_polar_df['Re'].min():.6g}; l'ensemble des vitesses autorisees reste-t-il hors domaine ? {diag_max_re < diag_polar_df['Re'].min()}") """

' diag_max_aws, diag_max_awa = apparent_wind(TWS_DIAG_MPS, diag_speed_bounds[1], TWA_DIAG_DEG, 0.0)\ndiag_max_re = C.RHO_AIR * diag_max_aws * diag_state.wing.mean_chord_m / C.MU_AIR\nprint(f"A la vitesse maximale autorisee ({diag_speed_bounds[1]:.6f} m/s), AWS={diag_max_aws:.6f} m/s et Re={diag_max_re:.6g}.")\nprint(f"La polaire commence a Re={diag_polar_df[\'Re\'].min():.6g}; l\'ensemble des vitesses autorisees reste-t-il hors domaine ? {diag_max_re < diag_polar_df[\'Re\'].min()}") '

In [ ]:
""" import trimesh

g = APP_STATE.get("geometry")
if g is None:
    print("Aucune géométrie chargée : relancer l'analyse.")
else:
    meshes = [
        ("coque brute", g.hull_mesh),
        ("coque immergée", g.immersed_hull_mesh),
        ("dérive", g.centerboard.mesh),
        ("safran", g.rudder.mesh),
    ]
    for name, mesh in meshes:
        print(
            name,
            "is_volume =", mesh.is_volume,
            "watertight =", mesh.is_watertight,
            "normales_coherentes =", mesh.is_winding_consistent,
            "volume =", safe_mesh_volume(mesh),
        ) """

' import trimesh\n\ng = APP_STATE.get("geometry")\nif g is None:\n    print("Aucune géométrie chargée : relancer l\'analyse.")\nelse:\n    meshes = [\n        ("coque brute", g.hull_mesh),\n        ("coque immergée", g.immersed_hull_mesh),\n        ("dérive", g.centerboard.mesh),\n        ("safran", g.rudder.mesh),\n    ]\n    for name, mesh in meshes:\n        print(\n            name,\n            "is_volume =", mesh.is_volume,\n            "watertight =", mesh.is_watertight,\n            "normales_coherentes =", mesh.is_winding_consistent,\n            "volume =", safe_mesh_volume(mesh),\n        ) '

In [ ]:
""" for name, mesh in meshes:
    print(name, "volume signé =", mesh.volume, "is_volume =", mesh.is_volume) """

' for name, mesh in meshes:\n    print(name, "volume signé =", mesh.volume, "is_volume =", mesh.is_volume) '

In [ ]:
""" for name, mesh in meshes:
    if mesh.is_watertight and mesh.volume < 0:
        mesh.invert() """

' for name, mesh in meshes:\n    if mesh.is_watertight and mesh.volume < 0:\n        mesh.invert() '

In [ ]:
""" G = masses.center_of_gravity
phi = np.deg2rad(20.0)
r = rotate_mesh_about_x(buoyancy_mesh, G, phi)
v0 = buoyancy_mesh.vertices[0] - G
v1 = r.vertices[0] - G
a0 = np.degrees(np.arctan2(v0[2], v0[1]))
a1 = np.degrees(np.arctan2(v1[2], v1[1]))
print(a1 - a0)   # doit valoir ±20, pas ±40 """

' G = masses.center_of_gravity\nphi = np.deg2rad(20.0)\nr = rotate_mesh_about_x(buoyancy_mesh, G, phi)\nv0 = buoyancy_mesh.vertices[0] - G\nv1 = r.vertices[0] - G\na0 = np.degrees(np.arctan2(v0[2], v0[1]))\na1 = np.degrees(np.arctan2(v1[2], v1[1]))\nprint(a1 - a0)   # doit valoir ±20, pas ±40 '